# LangChain & LangSmith 실습: 한국어 FAQ 챗봇 만들기

## 학습 목표
1. LangChain의 핵심 컴포넌트 이해 및 활용
2. RAG(Retrieval Augmented Generation) 시스템 구축
3. LangSmith를 통한 모니터링 및 디버깅: 로그인후에 API 키 생성
    - https://smith.langchain.com/
4. 실전 FAQ 챗봇 개발

## 시나리오
IT 회사 FAQ 챗봇: 직원들의 자주 묻는 질문에 자동으로 답변하는 시스템

## Step 1: 환경 설정

In [ ]:
# --------------------------
# 1. 필수 라이브러리 설치
# --------------------------

# LangChain: LLM 애플리케이션을 구축하기 위한 프레임워크
# - langchain: 핵심 프레임워크
# - langchain-openai: OpenAI 모델 통합
# - langchain-community: 커뮤니티 제공 확장 기능
# - langsmith: 애플리케이션 모니터링 및 디버깅 도구
!pip install langchain langchain-openai langchain-community langsmith

# 벡터 데이터베이스 및 유틸리티
# - chromadb: 임베딩 벡터를 저장하고 검색하는 벡터 데이터베이스
# - tiktoken: 텍스트를 토큰으로 분리하는 OpenAI의 토크나이저
# - python-dotenv: 환경 변수 관리
!pip install chromadb tiktoken
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepar

In [ ]:
# --------------------------
# 2. API 키 및 환경 변수 설정
# --------------------------

import os
from getpass import getpass

# 사용자로부터 OpenAI API 키를 안전하게 입력받아 환경 변수에 저장
# - 환경 변수에 저장하면 코드 전체에서 접근 가능
# - getpass()는 입력 내용을 화면에 표시하지 않아 보안에 유리
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# LangChain 추적 시스템 활성화 및 설정
# - LANGCHAIN_TRACING_V2="true": LangSmith 서버로 실행 로그를 전송
# - 이 로그를 통해 체인의 각 단계를 시각화하고 디버깅 가능
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# LangSmith API 키 입력 및 저장
# - LangSmith는 LangChain 애플리케이션의 모니터링 및 관리를 위한 플랫폼
# - API 키를 통해 사용자 인증 및 프로젝트 연동
os.environ["LANGCHAIN_API_KEY"] = getpass("LangSmith API Key: ")

# LangSmith 프로젝트 이름 설정
# - 이 이름으로 LangSmith 대시보드에서 프로젝트 식별
# - 모든 실행 기록이 이 프로젝트 아래에 저장됨
os.environ["LANGCHAIN_PROJECT"] = "korean-faq-chatbot-2"

# --------------------------
# 실행 흐름 설명:
# --------------------------
# 1. 먼저 필요한 모든 라이브러리를 설치합니다.
# 2. 사용자에게 OpenAI와 LangSmith의 API 키를 안전하게 입력받습니다.
# 3. 이 키들을 환경 변수에 저장하여 전역적으로 접근할 수 있게 합니다.
#
# 왜 이런 설정이 필요한가?
# - LangChain은 LLM과 상호작용하기 위해 OpenAI API 키가 필요합니다.
# - LangSmith는 애플리케이션 실행을 추적하고 분석하기 위해 별도의 API 키가 필요합니다.
#
# 구체적인 작동 과정:
# 1. !pip install: 패키지 관리자를 통해 파이썬 라이브러리를 설치
# 2. import os: 운영체제와 상호작용 (환경 변수 접근 등)
# 3. from getpass import getpass: 안전한 비밀번호 입력 함수 임포트
# 4. os.environ["KEY"] = value: 시스템 환경 변수에 키-값 쌍 저장
# 5. getpass("메시지"): 터미널에서 사용자 입력을 받되, 화면에 표시하지 않음
#
# 이 코드 블록이 실행되면:
# 1. 필요한 모든 패키지가 파이썬 환경에 설치됨
# 2. 터미널에서 두 번의 API 키 입력 요청이 뜸
# 3. 입력된 키가 메모리에 저장되어 이후 코드에서 사용됨
#
# 중요 개념:
# - 환경 변수: 운영체제 수준에서 관리하는 키-값 쌍으로, 여러 프로그램이 공유하는 설정
# - API 키: 외부 서비스(OpenAI, LangSmith)에 접근하기 위한 인증 토큰
# - 추적(Tracing): 애플리케이션 실행 과정의 각 단계를 기록하고 분석하는 기능

OpenAI API Key: ··········
LangSmith API Key: ··········


## Step 2: FAQ 데이터 준비

실제 회사에서 사용할 법한 HR FAQ 데이터를 준비합니다.

In [ ]:
# FAQ 데이터를 파이썬 리스트와 딕셔너리 구조로 정의합니다.
# 실제 운영환경에서는 파일(CSV, JSON)이나 데이터베이스에서 불러오는 것이 일반적입니다.
faq_data = [
    # 각 FAQ 항목은 딕셔너리 형태로 구성되며, "질문"과 "답변" 두 개의 키를 가집니다.
    {
        "질문": "연차는 어떻게 신청하나요?",
        "답변": "연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우 전화로 먼저 알리고 사후 처리도 가능합니다."
    },
    # 이하 9개의 FAQ 항목이 동일한 구조로 추가됩니다.
    {
        "질문": "월급날은 언제인가요?",
        "답변": "급여는 매월 25일에 지급됩니다. 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는 급여일 오전에 이메일로 발송됩니다."
    },
    {
        "질문": "재택근무 신청 방법을 알려주세요",
        "답변": "재택근무는 주 2회까지 가능합니다. 전날 오후 6시까지 팀장에게 슬랙으로 신청하면 됩니다. 재택근무 중에도 오전 10시-오후 5시는 업무 가능 상태여야 하며, 화상회의 참석이 가능해야 합니다."
    },
    {
        "질문": "건강검진은 언제 받나요?",
        "답변": "연간 건강검진은 입사 기념월에 받을 수 있습니다. 회사가 지정한 병원 리스트 중에서 선택 가능하며, 비용은 회사에서 전액 지원합니다. 인사팀으로 연락하시면 예약을 도와드립니다."
    },
    {
        "질문": "야근 수당은 어떻게 되나요?",
        "답변": "평일 오후 7시 이후 근무에 대해서는 야근 수당이 지급됩니다. 그룹웨어에서 야근 신청을 하고 팀장 승인을 받아야 합니다. 야근 수당은 시간당 기본급의 1.5배로 계산되며, 다음 달 급여에 포함됩니다."
    },
    {
        "질문": "경조사 휴가 기준은?",
        "답변": "경조사 휴가는 다음과 같이 제공됩니다. 본인 결혼: 5일, 자녀 결혼: 1일, 부모/배우자 부모 사망: 5일, 조부모/형제자매 사망: 3일. 경조사 발생 시 인사팀에 경조사 증빙서류를 제출해주세요."
    },
    {
        "질문": "점심시간은 몇 시부터 몇 시까지인가요?",
        "답변": "점심시간은 오후 12시부터 1시까지 1시간입니다. 팀별로 상황에 따라 11시 30분~1시 30분 사이에 탄력적으로 운영 가능합니다. 점심시간에는 자유롭게 외출 가능합니다."
    },
    {
        "질문": "사내 교육 프로그램이 있나요?",
        "답변": "분기별로 직무 교육과 온라인 교육 플랫폼을 제공합니다. 외부 교육이 필요한 경우 연간 100만원 한도 내에서 지원받을 수 있습니다. 교육 신청은 그룹웨어의 '교육신청' 메뉴에서 가능합니다."
    },
    {
        "질문": "주차는 어떻게 하나요?",
        "답변": "지하 주차장 이용이 가능하며, 입사 시 차량번호를 등록해야 합니다. 주차 공간이 한정되어 있어 선착순으로 배정됩니다. 주차 등록은 총무팀으로 신청하시면 됩니다."
    },
    {
        "질문": "복지 포인트는 어떻게 사용하나요?",
        "답변": "매년 1월 1일 기준으로 연간 150만원의 복지 포인트가 지급됩니다. 제휴 쇼핑몰에서 도서, 영화, 여행, 건강 등 다양한 분야에 사용 가능합니다. 사용 방법은 인사팀에서 발송하는 복지 가이드를 참조하세요."
    }
]

# FAQ 데이터의 개수를 확인하고 출력합니다.
# len(faq_data)는 리스트 faq_data의 요소 개수를 반환합니다.
print(f"총 {len(faq_data)}개의 FAQ 데이터가 준비되었습니다.")

# 예시로 첫 번째 FAQ 항목의 질문과 답변의 일부를 출력합니다.
print("\n예시:")  # \n은 줄바꿈 문자로 새로운 줄을 시작합니다.
print(f"Q: {faq_data[0]['질문']}")  # faq_data의 첫 번째 항목(인덱스 0)의 "질문" 키에 접근
print(f"A: {faq_data[0]['답변'][:50]}...")  # 첫 번째 항목의 "답변" 키에서 처음 50자만 잘라서 출력

# --------------------------
# 이 코드 블록의 목적:
# --------------------------
# 1. RAG 시스템에 필요한 지식 베이스(FAQ)를 메모리에 로드합니다.
# 2. 이 데이터는 나중에 벡터 데이터베이스에 저장되어 검색될 것입니다.
#
# 데이터 구조 설명:
# - faq_data: 파이썬 리스트
#   - 각 요소: 딕셔너리 (키: "질문", "답변")
#   - "질문": 직원들이 자주 묻는 질문 텍스트
#   - "답변": 해당 질문에 대한 공식 답변 텍스트
#
# 왜 이런 데이터 구조를 사용하나요?
# 1. 딕셔너리는 키-값 쌍으로 구조화된 데이터를 저장하기 적합합니다.
# 2. 리스트는 여러 개의 FAQ 항목을 순서대로 관리할 수 있습니다.
# 3. 나중에 이 데이터를 LangChain의 Document 객체로 변환하기 쉬운 형태입니다.
#
# 실행 흐름:
# 1. faq_data 리스트가 메모리에 생성됩니다.
# 2. print() 함수가 호출되어 데이터 개수와 예시를 출력합니다.
# 3. 사용자는 10개의 FAQ 항목이 준비되었음을 확인할 수 있습니다.
#
# 이 데이터는 다음 단계에서:
# 1. LangChain의 Document 객체로 변환됩니다.
# 2. 임베딩 모델을 통해 벡터로 변환됩니다.
# 3. Chroma 벡터 데이터베이스에 저장됩니다.
# 4. 사용자 질문이 들어오면 유사도 검색을 통해 관련 FAQ를 찾는 데 사용됩니다.
#
# 중요 개념:
# - 지식 베이스(Knowledge Base): 시스템이 참고할 수 있는 구조화된 정보 모음
# - RAG(Retrieval Augmented Generation): 외부 지식 베이스를 검색하여 LLM의 답변 품질을 높이는 기술
# - 임베딩(Embedding): 텍스트를 숫자 벡터로 변환하여 의미적 유사도를 계산할 수 있게 하는 기술

총 10개의 FAQ 데이터가 준비되었습니다.

예시:
Q: 연차는 어떻게 신청하나요?
A: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다. 최소 ...


## Step 3: 기본 LangChain 챗봇 (RAG 없이)

먼저 간단한 LLM 기반 챗봇을 만들어봅니다.

In [ ]:
# LangChain의 핵심 컴포넌트를 임포트합니다:
# - ChatOpenAI: OpenAI의 채팅 모델과 상호작용하기 위한 클래스
# - ChatPromptTemplate: LLM에 보낼 메시지를 구조화하는 템플릿
# - StrOutputParser: LLM의 출력을 문자열로 파싱하는 도구
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM(대형 언어 모델) 인스턴스를 초기화합니다.
# ChatOpenAI는 OpenAI의 API와 통신하는 래퍼 클래스입니다.
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 사용할 모델 지정 (OpenAI의 GPT-4o-mini 모델)
    temperature=0          # 응답의 창의성/무작위성 조절 (0: 일관된 답변, 1: 창의적인 답변)
    # temperature=0으로 설정하면 동일한 질문에 대해 항상 같은 답변이 나옵니다.
)

# 프롬프트 템플릿을 정의합니다. 프롬프트는 LLM에게 어떤 역할을 수행할지 지시하는 명령어입니다.
# 템플릿 문자열에서 {question}은 나중에 실제 질문으로 대체될 자리표시자입니다.
template = """당신은 친절한 HR 담당자입니다.
직원의 질문에 정확하고 친절하게 답변해주세요.

질문: {question}

답변:"""  # LLM이 이 위치에서부터 답변을 생성하기 시작합니다.

# ChatPromptTemplate.from_template() 메서드를 사용하여 템플릿 문자열을 ChatPromptTemplate 객체로 변환합니다.
# 이 객체는 나중에 실제 값으로 자리표시자를 대체할 수 있는 기능을 제공합니다.
prompt = ChatPromptTemplate.from_template(template)

# LangChain 체인을 구성합니다. 체인은 여러 컴포넌트를 연결한 실행 파이프라인입니다.
# 파이프 연산자(|)를 사용하여 컴포넌트들을 연결합니다. 이는 실제로 RunnableSequence 객체를 생성합니다.
basic_chain = prompt | llm | StrOutputParser()
# 실행 순서: 1. prompt -> 2. llm -> 3. StrOutputParser
# 1. prompt: 입력 데이터를 받아 프롬프트를 완성합니다 (자리표시자 {question}을 실제 질문으로 대체).
# 2. llm: 완성된 프롬프트를 OpenAI API로 전송하고 응답을 받아옵니다.
# 3. StrOutputParser: LLM의 응답(일반적으로 AIMessage 객체)에서 문자열 내용만 추출합니다.

# 체인을 테스트하기 위해 샘플 질문을 정의합니다.
question = "연차는 어떻게 신청하나요?"

# 체인을 실행합니다. invoke() 메서드는 체인에 입력을 전달하고 실행 결과를 반환합니다.
# 입력은 딕셔너리 형태로, 프롬프트 템플릿의 자리표시자 이름과 일치하는 키를 가져야 합니다.
response = basic_chain.invoke({"question": question})
# 실행 흐름:
# 1. {"question": "연차는 어떻게 신청하나요?"}가 prompt에 전달됨
# 2. prompt는 템플릿의 {question}을 "연차는 어떻게 신청하나요?"로 대체하여 완성된 프롬프트 생성
# 3. 완성된 프롬프트가 llm에 전달되고, OpenAI API 호출 발생
# 4. API 응답이 StrOutputParser에 전달되어 문자열로 변환
# 5. 변환된 문자열이 response 변수에 저장됨

# 결과를 출력합니다.
print(f"질문: {question}")
print(f"\n답변: {response}")

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: RAG 없이 순수 LLM만으로 FAQ 챗봇의 기본 동작을 구현하고 테스트합니다.
# 2. 작동 원리:
#    - 사용자 질문을 받아 미리 정의된 프롬프트 템플릿에 삽입합니다.
#    - 완성된 프롬프트를 LLM에 전달하여 응답을 생성합니다.
#    - LLM의 응답을 가공하여 사용자에게 보여줍니다.
#
# 한계점:
# 1. LLM은 FAQ 데이터에 없는 일반적인 지식으로만 답변합니다.
# 2. 회사의 구체적인 정책을 알지 못하므로 잘못된 정보를 제공할 수 있습니다.
# 3. 이 한계를 해결하기 위해 다음 단계에서 RAG(검색 증강 생성)를 도입합니다.
#
# LangChain 체인의 장점:
# 1. 모듈화: 프롬프트, LLM, 출력 파서를 독립적으로 변경 가능
# 2. 재사용성: 한 번 정의한 체인을 여러 번 호출 가능
# 3. 확장성: 더 많은 컴포넌트를 쉽게 추가할 수 있음
#
# 실행 결과 예상:
# - LLM은 HR 담당자 역할을 수행하며 질문에 답변합니다.
# - 하지만 FAQ 데이터를 학습하지 않았기 때문에, 실제 회사 정책과 다른 일반적인 답변을 할 수 있습니다.
# - 예: "연차 신청은 일반적으로 회사의 인사 시스템을 통해..." 같은 일반적인 답변
#
# 다음 단계에서 개선할 사항:
# 1. RAG 도입: FAQ 데이터를 검색하여 LLM의 컨텍스트로 제공
# 2. 더 정확한 답변: 회사 특정 정보를 반영한 답변 생성
# 3. 환각(Hallucination) 감소: 없는 정보를 만들지 않도록 안내

질문: 연차는 어떻게 신청하나요?

답변: 안녕하세요! 연차 신청 방법에 대해 안내해 드리겠습니다.

1. **신청서 작성**: 연차를 사용하고자 하는 날짜를 포함하여 연차 신청서를 작성해 주세요. 회사의 연차 신청 양식이 있다면 해당 양식을 사용하시면 됩니다.

2. **상사 승인**: 작성한 신청서를 상사에게 제출하여 승인을 받아야 합니다. 상사와의 대화에서 연차 사용 이유를 간단히 설명해 주시면 좋습니다.

3. **HR 부서 제출**: 상사의 승인을 받은 후, 최종 신청서를 HR 부서에 제출해 주세요. 이때, 제출 방법(이메일, 직접 제출 등)은 회사의 규정에 따라 다를 수 있으니 확인해 주세요.

4. **확인**: HR 부서에서 신청서를 확인한 후, 연차 사용이 승인되면 관련 내용을 안내해 드릴 것입니다.

추가로 궁금한 점이 있으시면 언제든지 문의해 주세요! 감사합니다.


### 🤔 문제점
- LLM이 일반적인 지식으로만 답변
- 회사 특정 정책을 모름
- 잘못된 정보를 제공할 수 있음 (환각 현상)

→ **RAG를 사용하여 해결!**

## Step 4: RAG 기반 FAQ 챗봇 구축

### 4.1 벡터 DB 생성

In [ ]:
# 임베딩 생성을 위한 OpenAI 임베딩 모델과 벡터 데이터베이스를 임포트합니다:
# - OpenAIEmbeddings: 텍스트를 벡터 임베딩으로 변환하는 모델
# - Chroma: 임베딩 벡터를 저장하고 검색하는 벡터 데이터베이스
# - Document: LangChain의 표준 문서 형식 (텍스트 내용과 메타데이터 포함)
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# FAQ 데이터를 LangChain의 Document 객체 형식으로 변환합니다.
# Document 객체는 page_content(본문)와 metadata(부가정보)를 가지는 표준화된 구조입니다.
documents = []  # 변환된 문서들을 저장할 빈 리스트 생성

# faq_data 리스트의 각 FAQ 항목을 순회하며 Document 객체로 변환
for faq in faq_data:
    # Document 객체 생성:
    # - page_content: FAQ의 질문과 답변을 하나의 문자열로 결합
    # - metadata: 문서의 출처와 원본 질문을 저장하는 딕셔너리
    doc = Document(
        page_content=f"질문: {faq['질문']}\n답변: {faq['답변']}",
        metadata={"source": "HR FAQ", "question": faq['질문']}
    )
    documents.append(doc)  # 생성된 Document 객체를 documents 리스트에 추가

# 변환된 문서의 개수를 출력
print(f"총 {len(documents)}개의 문서가 생성되었습니다.")

# 첫 번째 Document 객체의 내용을 예시로 출력
print("\n예시:")
print(documents[0].page_content)

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: FAQ 데이터를 LangChain 표준 형식(Document 객체)으로 변환하여
#    벡터 데이터베이스에 저장할 수 있는 형태로 준비합니다.
#
# 2. Document 객체 구조:
#    - page_content: 실제 검색될 텍스트 내용
#      형식: "질문: [질문 텍스트]\n답변: [답변 텍스트]"
#      이유: 질문과 답변을 함께 저장해야 검색 시 관련성을 높일 수 있음
#
#    - metadata: 문서에 대한 부가 정보 (검색 후 참조용)
#      - "source": 문서의 출처 식별 ("HR FAQ")
#      - "question": 원본 질문 텍스트 (답변 시 참조나 표시용)
#
# 3. 변환 과정의 상세 흐름:
#    a. documents = [] → 빈 리스트 초기화
#    b. for faq in faq_data: → faq_data의 각 항목(딕셔너리)에 대해 반복
#    c. doc = Document(...) → 현재 FAQ 항목을 Document 객체로 생성
#       1) page_content 구성:
#          - f"질문: {faq['질문']}\n답변: {faq['답변']}"
#          - 예: "질문: 연차는 어떻게 신청하나요?\n답변: 연차 신청은 사내 그룹웨어의..."
#       2) metadata 구성:
#          - {"source": "HR FAQ", "question": faq['질문']}
#          - source: 모든 문서에 동일하게 "HR FAQ" 지정
#          - question: 현재 FAQ의 질문 텍스트 저장
#    d. documents.append(doc) → 생성된 Document 객체를 리스트에 추가
#    e. 모든 FAQ 항목에 대해 c~d 과정 반복
#
# 4. 변환 결과:
#    - documents 리스트에는 10개의 Document 객체가 순서대로 저장됨
#    - 각 Document 객체는 page_content와 metadata를 가짐
#
# 5. 출력 결과 예시:
#    총 10개의 문서가 생성되었습니다.
#
#    예시:
#    질문: 연차는 어떻게 신청하나요?
#    답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다...
#
# 6. 왜 이런 변환이 필요한가?
#    - 벡터 데이터베이스(Chroma)는 Document 객체 형식을 기대함
#    - Document 형식은 텍스트(content)와 메타데이터를 구조화하여 저장
#    - 메타데이터는 검색 후 문서의 출처를 추적하거나 추가 정보를 표시하는 데 사용
#
# 7. 메타데이터의 중요성:
#    - "question" 필드: 검색된 문서가 어떤 질문에 대한 답변인지 식별
#    - "source" 필드: 문서의 신뢰성과 카테고리 식별
#    - 나중에 RAG 체인에서 검색된 문서를 사용할 때 메타데이터를 참조 가능
#
# 8. 다음 단계에서 이 documents 리스트는:
#    - OpenAIEmbeddings 모델을 통해 벡터 임베딩으로 변환됨
#    - Chroma 벡터 데이터베이스에 저장됨
#    - 사용자 질문과의 유사도 검색에 사용됨
#
# 중요 개념:
# - 임베딩(Embedding): 텍스트를 고정된 길이의 숫자 벡터로 변환
# - 벡터 데이터베이스: 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스
# - 유사도 검색: 쿼리 벡터와 저장된 벡터들 사이의 코사인 유사도 등을 계산

총 10개의 문서가 생성되었습니다.

예시:
질문: 연차는 어떻게 신청하나요?
답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우 전화로 먼저 알리고 사후 처리도 가능합니다.


In [ ]:
# OpenAI의 임베딩 모델을 초기화합니다. 이 모델은 텍스트를 숫자 벡터로 변환하는 역할을 합니다.
# OpenAIEmbeddings 클래스는 OpenAI API를 사용하여 텍스트 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 매개변수:
# - model="text-embedding-3-small": 사용할 임베딩 모델 지정 (OpenAI의 text-embedding-3-small 모델)
#   이 모델은 상대적으로 작은 차원(1536차원)의 임베딩을 생성하며 비용 효율적입니다.
#   텍스트 임베딩은 텍스트의 의미를 숫자 벡터로 표현하여, 유사한 의미의 텍스트는 유사한 벡터를 갖습니다.

# 벡터 데이터베이스를 생성합니다. Chroma는 로컬에서 실행되는 오픈소스 벡터 데이터베이스입니다.
# from_documents 메서드는 문서 목록을 받아 자동으로 임베딩을 생성하고 벡터 DB에 저장합니다.
vectorstore = Chroma.from_documents(
    # 매개변수:
    documents=documents,           # 이전 단계에서 생성한 Document 객체들의 리스트
    embedding=embeddings,          # 위에서 초기화한 임베딩 모델
    collection_name="hr_faq"       # 벡터 DB 내에서 이 문서들을 저장할 컬렉션 이름
    # 컬렉션(collection)은 관계형 데이터베이스의 테이블과 유사한 개념으로,
    # 특정 주제나 유형의 문서들을 그룹화하는 데 사용됩니다.
)

# 생성 완료 메시지 출력
print(" 벡터 DB가 생성되었습니다!")

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: FAQ 문서들을 벡터 형식으로 변환하여 유사도 검색이 가능한 데이터베이스에 저장합니다.
#
# 2. 작동 원리 - 단계별 상세 설명:
#    a. OpenAIEmbeddings 초기화:
#       1) embeddings 객체가 생성되면, 이 객체는 텍스트를 벡터로 변환하는 기능을 제공합니다.
#       2) 내부적으로는 OpenAI API에 텍스트를 전송하고, 임베딩 벡터(숫자 배열)를 반환받습니다.
#
#    b. Chroma.from_documents 실행:
#       1) Chroma는 내부적으로 다음 작업을 순차적으로 수행합니다:
#          1단계: 각 Document 객체의 page_content 텍스트를 추출합니다.
#          2단계: embeddings 객체를 사용해 각 텍스트를 임베딩 벡터로 변환합니다.
#                 이때 OpenAI API가 호출되어 각 텍스트에 대한 벡터가 생성됩니다.
#          3단계: 변환된 벡터와 원본 문서(메타데이터 포함)를 로컬 디스크에 저장합니다.
#                 기본 저장 경로는 "./chroma_db/" 디렉토리입니다.
#
# 3. 임베딩 과정의 기술적 이해:
#    - 임베딩: 자연어 텍스트를 고정된 길이의 실수 벡터(예: 1536차원)로 변환하는 과정
#    - 예시: "연차 신청 방법" → [0.123, -0.456, 0.789, ...] (1536개의 숫자)
#    - 의미론적 유사성: 유사한 의미의 텍스트는 벡터 공간에서 가까운 위치에 매핑됩니다.
#
# 4. 벡터 데이터베이스의 구조:
#    - 컬렉션(collection): "hr_faq"라는 이름의 논리적 문서 그룹
#    - 각 문서는 다음으로 구성됨:
#        * 고유 ID
#        * 임베딩 벡터 (1536차원)
#        * 원본 텍스트 (page_content)
#        * 메타데이터 (source, question)
#
# 5. 실제 내부 작업 흐름:
#    a. for doc in documents:
#          text = doc.page_content  # "질문: ...\n답변: ..."
#          vector = embeddings.embed_query(text)  # API 호출, 1536차원 벡터 반환
#          Chroma는 (vector, text, metadata)를 저장
#
#    b. 이 과정이 documents 리스트의 모든 문서(10개)에 대해 반복됩니다.
#    c. API 호출은 10번 발생합니다(문서당 1회).
#
# 6. 생성된 벡터 DB의 특징:
#    - 로컬 파일 시스템에 저장되므로 인터넷 연결 없이도 검색 가능
#    - 자동으로 인덱스 생성되어 빠른 유사도 검색 지원
#    - 메타데이터 필터링 등의 고급 검색 기능 제공
#
# 7. 출력 결과:
#    - "벡터 DB가 생성되었습니다!" 메시지가 표시됩니다.
#    - 실제로는 ./chroma_db/ 디렉토리에 데이터 파일들이 생성됩니다.
#
# 8. 왜 벡터 DB가 필요한가?
#    - RAG 시스템에서 사용자 질문과 가장 관련성 높은 문서를 빠르게 검색하기 위해
#    - 유사도 계산: 사용자 질문을 임베딩한 후, 저장된 문서 벡터들과 유사도(코사인 유사도) 계산
#    - 검색 속도: 수천~수만 개의 문서에서도 밀리초 단위로 검색 가능
#
# 9. 다음 단계에서 벡터 DB는:
#    - vectorstore.as_retriever() 메서드로 검색기(Retriever)로 변환됩니다.
#    - 사용자 질문이 들어오면, 질문을 임베딩하여 가장 유사한 FAQ 문서들을 검색합니다.
#    - 검색된 문서들은 LLM의 컨텍스트로 제공되어 정확한 답변 생성에 활용됩니다.
#
# 중요 개념:
# - 임베딩 벡터: 텍스트의 의미를 수치화한 표현
# - 벡터 검색: 쿼리 벡터와 문서 벡터들 사이의 유사도 계산을 통해 관련 문서 찾기
# - 코사인 유사도: 벡터 간 각도의 코사인 값을 유사도 점수로 사용 (1: 완전 동일, -1: 완전 반대)

 벡터 DB가 생성되었습니다!


### 4.2 검색 테스트

In [ ]:
# 벡터 데이터베이스에서 유사도 검색을 테스트합니다.
# 사용자의 자연어 질문과 가장 유사한 의미를 가진 FAQ 문서들을 검색하는 과정입니다.

# 검색할 쿼리(질문)를 정의합니다.
query = "휴가 신청은 어떻게 해요?"
# 이 쿼리는 사용자가 실제로 입력할 수 있는 자연어 질문입니다.
# "휴가 신청"은 FAQ에 있는 "연차 신청"과 유사한 의미를 가집니다.

# vectorstore의 similarity_search 메서드를 호출하여 유사도 검색을 수행합니다.
# 매개변수:
# - query: 검색할 텍스트 쿼리
# - k=3: 반환할 문서의 개수 (상위 3개)
results = vectorstore.similarity_search(query, k=3)
# similarity_search의 내부 동작:
# 1. query 텍스트("휴가 신청은 어떻게 해요?")를 임베딩 벡터로 변환
#    - embeddings 객체를 사용해 OpenAI API 호출 발생
#    - 쿼리 텍스트가 1536차원 벡터로 변환됨
# 2. 변환된 쿼리 벡터와 저장된 모든 문서 벡터 간의 유사도 계산
#    - 일반적으로 코사인 유사도(Cosine Similarity) 사용
#    - 유사도 점수는 -1(반대)에서 1(동일) 사이의 값
# 3. 유사도 점수가 높은 순으로 상위 k개(여기서는 3개) 문서 선택
# 4. 선택된 문서들의 page_content와 metadata를 포함한 Document 객체 반환

# 검색 결과 출력
print(f"검색 쿼리: {query}\n")
print("검색 결과:")

# results는 Document 객체들의 리스트입니다. enumerate를 사용하여 인덱스와 함께 반복합니다.
# enumerate(results, 1): 인덱스를 1부터 시작하게 설정 (사용자 친화적인 표시)
for i, doc in enumerate(results, 1):
    print(f"\n[결과 {i}]")  # 현재 결과의 순번 출력
    print(doc.page_content)  # Document의 본문 내용 출력
    print(f"메타데이터: {doc.metadata}")  # Document의 메타데이터 출력

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: 벡터 데이터베이스가 올바르게 구축되었는지 검증하고,
#    사용자 질문과 FAQ 문서 간의 의미론적 유사도 검색이 어떻게 작동하는지 확인합니다.
#
# 2. 작동 원리 - 상세 기술적 과정:
#    a. 쿼리 임베딩 생성:
#       1) query 문자열이 OpenAIEmbeddings 객체에 전달됨
#       2) embeddings.embed_query(query)가 내부적으로 호출됨
#       3) OpenAI API로 쿼리 텍스트 전송 → 임베딩 벡터(1536차원) 반환
#
#    b. 유사도 계산 및 검색:
#       1) 쿼리 벡터와 저장된 모든 문서 벡터 간의 유사도 계산
#          - 각 문서 벡터와 쿼리 벡터의 코사인 유사도 계산
#          - 공식: similarity = (A·B) / (||A|| * ||B||)
#          - 여기서 A는 쿼리 벡터, B는 문서 벡터
#       2) 계산된 유사도 점수를 기준으로 내림차순 정렬
#       3) 상위 3개 문서 선택
#
# 3. 유사도 검색의 의미론적 이해:
#    - 쿼리: "휴가 신청은 어떻게 해요?"
#    - FAQ 문서: "질문: 연차는 어떻게 신청하나요?..."
#    - 두 텍스트는 표면적으로 다른 단어를 사용했지만 의미는 매우 유사함
#    - 임베딩 모델은 이를 이해하고 높은 유사도 점수를 부여함
#
# 4. 출력 결과 예상:
#    검색 쿼리: 휴가 신청은 어떻게 해요?
#
#    검색 결과:
#
#    [결과 1]
#    질문: 연차는 어떻게 신청하나요?
#    답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다...
#    메타데이터: {'source': 'HR FAQ', 'question': '연차는 어떻게 신청하나요?'}
#
#    [결과 2]
#    (두 번째로 유사한 FAQ, 예: 재택근무 신청 방법)
#
#    [결과 3]
#    (세 번째로 유사한 FAQ)
#
# 5. 검색 결과의 품질 평가 요소:
#    - 관련성: "연차 신청" FAQ가 1순위로 검색되는가?
#    - 의미 이해: "휴가"와 "연차"의 동의어 관계를 이해하는가?
#    - 유용성: 검색된 문서가 사용자 질문에 대한 답변을 제공하는가?
#
# 6. similarity_search 메서드의 추가 옵션 (여기서는 사용하지 않음):
#    - filter: 메타데이터 조건으로 검색 결과 필터링
#    - score_threshold: 특정 유사도 점수 이상의 결과만 반환
#
# 7. 코사인 유사도의 수학적 이해:
#    - 두 벡터의 방향이 얼마나 일치하는지 측정
#    - 길이(크기)는 무시하고 방향만 고려
#    - 텍스트 임베딩에서 의미적 유사도를 측정하는 데 효과적
#
# 8. 이 테스트의 중요성:
#    - 벡터 DB 구축이 성공적으로 완료되었는지 확인
#    - 임베딩 모델이 한국어 텍스트의 의미를 잘 이해하는지 검증
#    - RAG 시스템의 검색 단계가 정상 작동하는지 확인
#
# 9. 실제 어플리케이션에서의 활용:
#    - 사용자가 "휴가 신청"이라고 물어도 시스템은 "연차 신청" FAQ를 찾아낼 수 있음
#    - 동의어, 유의어, 관련어 검색이 가능해짐
#    - 키워드 매칭이 아닌 의미 기반 검색이 이루어짐
#
# 10. 다음 단계에서의 활용:
#     - 검색된 문서들은 RAG 체인에서 LLM의 컨텍스트로 사용됨
#     - LLM은 검색된 FAQ 정보를 바탕으로 정확한 답변 생성
#     - vectorstore.as_retriever()로 검색기를 만들어 체인에 통합
#
# 중요 개념:
# - 의미론적 검색(Semantic Search): 단어 매칭이 아닌 의미 유사도 기반 검색
# - 임베딩 공간(Embedding Space): 모든 텍스트가 벡터로 매핑된 고차원 공간
# - nearest neighbor search: 쿼리 벡터와 가장 가까운(유사한) 벡터 찾기

검색 쿼리: 휴가 신청은 어떻게 해요?

검색 결과:

[결과 1]
질문: 연차는 어떻게 신청하나요?
답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우 전화로 먼저 알리고 사후 처리도 가능합니다.
메타데이터: {'source': 'HR FAQ', 'question': '연차는 어떻게 신청하나요?'}

[결과 2]
질문: 경조사 휴가 기준은?
답변: 경조사 휴가는 다음과 같이 제공됩니다. 본인 결혼: 5일, 자녀 결혼: 1일, 부모/배우자 부모 사망: 5일, 조부모/형제자매 사망: 3일. 경조사 발생 시 인사팀에 경조사 증빙서류를 제출해주세요.
메타데이터: {'question': '경조사 휴가 기준은?', 'source': 'HR FAQ'}

[결과 3]
질문: 주차는 어떻게 하나요?
답변: 지하 주차장 이용이 가능하며, 입사 시 차량번호를 등록해야 합니다. 주차 공간이 한정되어 있어 선착순으로 배정됩니다. 주차 등록은 총무팀으로 신청하시면 됩니다.
메타데이터: {'question': '주차는 어떻게 하나요?', 'source': 'HR FAQ'}


### 4.3 RAG 체인 구성

In [ ]:
# LangChain의 최신 스타일(2025)로 RAG 체인을 구성합니다.
# 필요한 클래스와 함수를 임포트합니다:
# - ChatPromptTemplate: LLM 프롬프트를 구조화하기 위한 템플릿
# - RunnableParallel, RunnablePassthrough: 체인 구성 요소들을 조합하기 위한 Runnable 클래스들
# - StrOutputParser: LLM 출력을 문자열로 파싱
# - ChatOpenAI: OpenAI 채팅 모델 (이전에 이미 임포트되어 재사용)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI  # 주석: 이미 이전에 임포트되어 사용 중

# 벡터 저장소를 검색기(Retriever)로 변환합니다.
# 검색기는 쿼리를 받아 관련 문서를 검색하는 인터페이스를 제공합니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# as_retriever() 메서드: Chroma 벡터 저장소를 검색기 객체로 변환
# search_kwargs={"k": 3}: 검색 시 상위 3개의 문서를 반환하도록 설정
# 검색기는 내부적으로 vectorstore.similarity_search()를 호출합니다.

# RAG 프롬프트 템플릿을 정의합니다.
# 이 템플릿은 LLM에게 검색된 FAQ 정보를 바탕으로 답변을 생성하도록 지시합니다.
rag_template = """당신은 회사의 HR 챗봇입니다.
아래 제공된 FAQ 정보를 바탕으로 직원의 질문에 정확하고 친절하게 답변해주세요.

FAQ 정보:
{context}

질문: {question}

답변 가이드:
1. FAQ에 정확한 답변이 있으면 그대로 전달하세요
2. FAQ에 없는 내용이면 "정확한 정보는 인사팀(내선 1234)으로 문의해주세요"라고 안내하세요
3. 친절하고 전문적인 톤을 유지하세요

답변:"""
# {context}: 검색기(retriever)가 반환한 FAQ 문서들이 이 자리에 삽입될 자리표시자
# {question}: 사용자의 원본 질문이 삽입될 자리표시자

# 템플릿 문자열을 ChatPromptTemplate 객체로 변환합니다.
rag_prompt = ChatPromptTemplate.from_template(rag_template)

# 답변만 생성하는 RAG 체인을 LCEL(LangChain Expression Language) 스타일로 구성합니다.
# LCEL은 파이프(|) 연산자를 사용하여 컴포넌트들을 연결하는 선언적 API입니다.
answer_chain = (
    # RunnableParallel: 여러 입력을 병렬로 처리하는 컴포넌트
    # 이 단계에서는 두 가지 작업을 병렬로 실행:
    # 1. context 생성: retriever를 사용해 관련 문서 검색
    # 2. question 전달: 원본 질문을 그대로 다음 단계로 전달
    RunnableParallel(
        # context: retriever에 질문을 전달하여 관련 문서 검색
        # retriever는 함수처럼 동작하여 입력(질문)을 받고 문서 리스트를 반환
        context=retriever,                # 질문을 retriever에 전달 -> 문서 검색 -> context 생성
        # question: 원본 질문을 변경 없이 다음 단계로 전달
        # RunnablePassthrough()는 입력을 그대로 출력으로 전달하는 통과 컴포넌트
        question=RunnablePassthrough()    # 질문은 그대로 통과시켜 다음 단계로 전달
    )
    # 파이프(|) 연산자로 체인 연결: 이전 단계의 출력이 다음 단계의 입력이 됨
    | rag_prompt     # 병렬 처리 결과를 rag_prompt에 전달하여 프롬프트 완성
    | llm            # 완성된 프롬프트를 LLM에 전달하여 답변 생성
    | StrOutputParser()  # LLM의 출력을 문자열로 파싱
)
# answer_chain의 전체 실행 흐름:
# 1. 입력: 사용자 질문 문자열
# 2. RunnableParallel 실행:
#    - context 경로: 질문 → retriever → 관련 문서 리스트
#    - question 경로: 질문 → RunnablePassthrough → 원본 질문
# 3. rag_prompt: {context}에 문서 리스트, {question}에 원본 질문 삽입
# 4. llm: 완성된 프롬프트를 OpenAI API로 전송, 응답 받음
# 5. StrOutputParser: LLM 응답을 문자열로 변환
# 6. 출력: 최종 답변 문자열

# "답변 + 출처 문서" 둘 다 반환하는 체인을 생성합니다.
# 이 체인은 답변과 검색된 문서 원본을 함께 반환하여 투명성과 디버깅을 제공합니다.
qa_chain = RunnableParallel(
    # answer: answer_chain을 실행하여 답변 문자열 생성
    answer=answer_chain,
    # source_documents: retriever를 다시 실행하여 검색된 문서 원본 반환
    # (answer_chain 내부에서 이미 retriever가 실행되지만, 여기서는 원본 문서를 따로 저장)
    source_documents=retriever
)
# qa_chain의 실행 흐름:
# 1. 입력: 사용자 질문 문자열
# 2. RunnableParallel 실행 (두 경로 병렬 처리):
#    - answer 경로: 질문 → answer_chain → 답변 문자열
#    - source_documents 경로: 질문 → retriever → 문서 리스트
# 3. 출력: {"answer": 답변문자열, "source_documents": 문서리스트} 형식의 딕셔너리

print("=== RAG 체인이 생성되었습니다!")

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: LangChain의 최신 Runnable 인터페이스를 사용하여 RAG 체인을 구성합니다.
#
# 2. Runnable 인터페이스의 장점:
#    - 선언적 프로그래밍: 체인 구조를 명확하게 표현
#    - 병렬 처리: 여러 작업을 동시에 실행 가능
#    - 재사용성: 컴포넌트들을 쉽게 재조합 가능
#    - 디버깅: 각 단계의 입력/출력을 명확하게 추적 가능
#
# 3. retriever의 역할 상세 설명:
#    - vectorstore.as_retriever(): 벡터 저장소를 검색 인터페이스로 변환
#    - retriever는 호출 가능한 객체: retriever.invoke(question) 형태로 사용
#    - 내부 동작:
#        1) question을 임베딩 벡터로 변환
#        2) 벡터 저장소에서 유사도 검색 수행
#        3) 상위 k개의 Document 객체 반환
#
# 4. RunnableParallel의 작동 원리:
#    - 딕셔너리 형태로 병렬 작업 정의: {"output_key": runnable_component}
#    - 모든 병렬 작업이 동시에 실행됨
#    - 각 작업의 결과가 지정된 output_key에 저장됨
#    - 최종 출력: {"context": 문서리스트, "question": 원본질문} 형식의 딕셔너리
#
# 5. RunnablePassthrough의 역할:
#    - 입력 값을 변경 없이 출력으로 전달하는 통과 컴포넌트
#    - 복잡한 체인에서 특정 값을 보존할 때 유용
#
# 6. 프롬프트 템플릿의 중요성:
#    - {context}: 검색된 FAQ 문서들이 하나의 문자열로 결합되어 삽입됨
#    - {question}: 사용자의 원본 질문이 그대로 삽입됨
#    - 답변 가이드: LLM의 응답 품질을 제어하는 중요한 지시사항
#
# 7. answer_chain과 qa_chain의 차이:
#    - answer_chain: 최종 답변 문자열만 반환 (단순 챗봇 용도)
#    - qa_chain: 답변과 출처 문서를 함께 반환 (디버깅, 투명성, 고급 기능용)
#
# 8. 체인 실행 시 데이터 흐름:
#    사용자 질문 "연차 신청 방법?"
#    ↓
#    retriever: "연차 신청 방법?" → 임베딩 → 유사도 검색 → 관련 FAQ 문서 3개
#    ↓
#    rag_prompt:
#        context에 FAQ 문서들 삽입, question에 "연차 신청 방법?" 삽입
#        → 완성된 프롬프트 생성
#    ↓
#    llm: 프롬프트를 API로 전송 → 응답 생성
#    ↓
#    StrOutputParser: 응답에서 텍스트 추출 → 답변 문자열
#
# 9. RAG 체인의 핵심 가치:
#    - 정확성: FAQ 데이터베이스의 정확한 정보를 기반으로 답변
#    - 환각 방지: 모르는 질문에 대해 "인사팀 문의"로 안전하게 응답
#    - 일관성: 동일한 질문에 대해 항상 동일한 답변 제공
#
# 10. LangSmith 통합 (환경 변수로 이미 설정됨):
#     - 이 체인이 실행될 때마다 LangSmith 서버로 로그가 전송됨
#     - 각 단계의 입력/출력, 소요 시간, 토큰 사용량 등을 모니터링 가능
#
# 중요 개념:
# - LCEL (LangChain Expression Language): 체인을 구성하기 위한 선언적 API
# - Runnable: LangChain에서 모든 체인 구성 요소의 기본 인터페이스
# - Retriever: 검색 기능을 추상화한 인터페이스
# - RAG 체인: 검색(Retrieval) + 생성(Generation)의 파이프라인

=== RAG 체인이 생성되었습니다!


In [ ]:
# RAG 체인을 테스트 실행하여 실제로 어떻게 동작하는지 확인합니다.
# 사용자 질문을 정의합니다. 이 질문은 자연스러운 표현으로 FAQ의 내용을 묻고 있습니다.
query = "연차에 대해 알고 싶어"
# 이 질문은 "연차 신청 방법"보다 더 포괄적이고 자연스러운 표현입니다.
# RAG 시스템은 이를 이해하고 관련 FAQ를 검색해야 합니다.

# qa_chain을 호출(실행)하여 질문에 대한 답변과 검색된 문서를 얻습니다.
# invoke() 메서드는 체인에 입력을 전달하고 실행 결과를 반환합니다.
# qa_chain은 이전에 정의한대로 RunnableParallel로 구성되어 있습니다.
result = qa_chain.invoke(query)
# 실행 과정 상세:
# 1. qa_chain.invoke(query)가 호출되면 내부적으로 다음이 동시에 실행됨:
#    a) answer_chain 경로: query → answer_chain → 답변 문자열 생성
#    b) source_documents 경로: query → retriever → 관련 문서 리스트 검색
# 2. 결과는 딕셔너리 형태로 반환됨:
#    {"answer": "생성된 답변 문자열", "source_documents": [Document1, Document2, Document3]}

# 테스트 결과를 출력합니다.
print("--- 답변:")
# result 딕셔너리에서 "answer" 키의 값을 가져와 출력합니다.
# 이 값은 answer_chain이 생성한 최종 답변 문자열입니다.
print(result["answer"])

print("\n--- 사용된 문서 수:", len(result["source_documents"]))
# result["source_documents"]는 검색된 Document 객체들의 리스트입니다.
# len() 함수를 사용하여 리스트의 길이(문서 개수)를 계산합니다.
# retriever는 search_kwargs={"k": 3}으로 설정되었으므로 최대 3개의 문서가 반환됩니다.

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: 구축한 RAG 체인이 실제로 정상 작동하는지 확인하고,
#    답변 생성과 문서 검색이 어떻게 연동되는지 이해합니다.
#
# 2. 작동 원리 - qa_chain.invoke(query)의 상세 실행 과정:
#    Step 1: qa_chain은 RunnableParallel이므로 두 경로를 병렬로 실행 시작
#
#    Step 2-1: answer_chain 경로 실행
#        a) RunnableParallel(context=retriever, question=RunnablePassthrough()) 실행:
#           - 입력 query가 retriever와 RunnablePassthrough에 동시에 전달됨
#           - context 경로: query → retriever → 검색 수행 → 문서 리스트 반환
#           - question 경로: query → RunnablePassthrough → query 그대로 통과
#           - 결과: {"context": [Document1, Document2, Document3], "question": "연차에 대해 알고 싶어"}
#
#        b) rag_prompt 실행:
#           - 입력: {"context": 문서리스트, "question": "연차에 대해 알고 싶어"}
#           - rag_template의 {context} 자리에 문서 리스트의 내용이 삽입됨
#             (문서 리스트의 page_content들이 문자열로 결합되어 삽입)
#           - rag_template의 {question} 자리에 "연차에 대해 알고 싶어" 삽입
#           - 완성된 프롬프트 예시:
#             """
#             당신은 회사의 HR 챗봇입니다.
#             아래 제공된 FAQ 정보를 바탕으로 직원의 질문에 정확하고 친절하게 답변해주세요.
#
#             FAQ 정보:
#             질문: 연차는 어떻게 신청하나요?
#             답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서...
#
#             (다른 문서들)
#
#             질문: 연차에 대해 알고 싶어
#
#             답변 가이드:...
#             답변:
#             """
#
#        c) llm 실행:
#           - 완성된 프롬프트가 OpenAI API로 전송됨
#           - GPT-4o-mini 모델이 프롬프트를 처리하고 답변 생성
#           - 출력: AIMessage 객체 (텍스트 답변 포함)
#
#        d) StrOutputParser 실행:
#           - AIMessage 객체에서 텍스트 내용만 추출
#           - 최종 출력: "연차 신청은 사내 그룹웨어의..." 같은 답변 문자열
#
#    Step 2-2: source_documents 경로 실행 (answer_chain과 병렬)
#        - query → retriever → 검색 수행 → 문서 리스트 반환
#        - 이 retriever 호출은 answer_chain 내부의 retriever 호출과 완전히 동일함
#          (같은 쿼리, 같은 설정이므로 동일한 문서 리스트 반환)
#
#    Step 3: qa_chain 결과 통합
#        - answer_chain의 출력: 답변 문자열
#        - source_documents 경로의 출력: 문서 리스트
#        - 최종 결과: {"answer": "답변문자열", "source_documents": [Document1, Document2, Document3]}
#
# 3. 출력 결과 예상:
#    --- 답변:
#    연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하면 됩니다...
#    (FAQ 정보를 바탕으로 생성된 정확한 답변)
#
#    --- 사용된 문서 수: 3
#
# 4. RAG의 핵심 장점이 드러나는 부분:
#    - 자연어 이해: "연차에 대해 알고 싶어" → "연차는 어떻게 신청하나요?" FAQ와 연결
#    - 정확한 정보 제공: LLM이 FAQ 데이터베이스의 정확한 정보를 바탕으로 답변
#    - 환각 방지: FAQ에 없는 정보는 "인사팀 문의"로 안전하게 응답
#
# 5. retriever가 두 번 호출되는 이유와 효율성:
#    - answer_chain 내부에서 한 번, source_documents에서 한 번 총 두 번 호출됨
#    - 실제로는 같은 쿼리에 대해 두 번 검색하므로 비효율적으로 보일 수 있음
#    - 하지만:
#        1) 각 호출은 독립적이므로 캐싱 등의 최적화 가능
#        2) 실제 운영환경에서는 검색 결과를 재사용하도록 최적화할 수 있음
#        3) 디버깅과 투명성을 위해 원본 문서를 별도로 저장하는 것이 유용함
#
# 6. LangSmith에서 이 실행의 모니터링:
#    - qa_chain.invoke()가 호출되면 LangChain 추적 시스템이 자동으로 활성화됨
#    - LangSmith 대시보드에서 다음을 확인할 수 있음:
#        * 전체 체인 실행 과정
#        * 각 단계별 소요 시간
#        * retriever가 검색한 문서들
#        * LLM에 전송된 프롬프트와 받은 응답
#        * 사용된 토큰 수와 비용
#
# 7. 테스트 질문의 의도:
#    - "연차에 대해 알고 싶어"는 불완전한 질문처럼 보이지만 RAG 시스템이 처리 가능
#    - 사용자가 정확한 질문을 모를 때도 자연스럽게 정보를 요청할 수 있음
#    - RAG 시스템은 쿼리의 의미를 이해하고 가장 관련성 높은 FAQ를 검색함
#
# 8. 다음 단계에서의 활용:
#    - 이 테스트를 바탕으로 더 다양한 질문에 대한 응답 테스트 가능
#    - 검색된 문서들을 분석하여 RAG 시스템의 품질 평가
#    - 필요시 retriever의 k 값을 조정하거나 검색 방식을 변경
#
# 중요 개념:
# - 체인 호출(Chain Invocation): 구성된 파이프라인에 입력을 주고 출력을 받는 실행 과정
# - 병렬 실행(Parallel Execution): 여러 작업이 동시에 실행되어 전체 처리 시간 단축
# - 결과 추적(Result Tracing): 각 실행 단계의 입력/출력을 기록하여 디버깅 가능

--- 답변:
연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우에는 전화로 먼저 알리고 사후 처리도 가능합니다. 추가로 궁금한 점이 있으시면 언제든지 문의해 주세요!

--- 사용된 문서 수: 3


### 4.4 RAG 챗봇 테스트

In [ ]:
# FAQ 챗봇에게 질문을 하고 결과를 포맷팅하여 출력하는 함수를 정의합니다.
def ask_faq(question):
    """FAQ 챗봇에게 질문하기"""  # 함수의 간단한 설명 (docstring)

    # RAG 체인을 실행하여 질문에 대한 결과를 얻습니다.
    # qa_chain은 이전에 정의한 RunnableParallel 체인으로,
    # answer(답변)와 source_documents(검색된 문서)를 반환합니다.
    result = qa_chain.invoke(question)  # 주석: query를 딕셔너리 형태로 감싸지 않고 직접 문자열 전달
    # 이전 버전의 LangChain에서는 {"query": question} 형태로 전달해야 했지만,
    # 최신 버전에서는 질문 문자열을 그대로 전달할 수 있습니다.

    # 결과에서 답변과 검색된 문서를 추출합니다.
    answer = result["answer"]  # qa_chain이 생성한 답변 문자열
    sources = result["source_documents"]  # 검색된 Document 객체들의 리스트

    # 결과를 사용자 친화적인 형식으로 출력합니다.
    print(f"💬 질문: {question}")  # 사용자의 원본 질문 출력 (이모지로 시각적 강조)
    print(f"\n🤖 답변: {answer}")  # 생성된 답변 출력 (로봇 이모지)
    print(f"\n📚 참고한 문서 수: {len(sources)}")  # 검색된 문서의 개수 출력 (책 이모지)

    # 검색된 문서가 있을 경우, 각 문서의 메타데이터에서 FAQ 질문을 추출하여 출력합니다.
    if len(sources) > 0:
        print("\n관련 FAQ:")  # 관련 FAQ 목록 헤더
        # enumerate를 사용하여 문서 리스트를 순회하며 인덱스(1부터 시작)와 문서를 얻습니다.
        for i, doc in enumerate(sources, 1):
            # 문서의 메타데이터에서 "question" 키의 값을 가져옵니다.
            # get() 메서드는 키가 없을 경우 기본값 "(질문 정보 없음)"을 반환합니다.
            q = doc.metadata.get("question", "(질문 정보 없음)")
            # 각 FAQ 질문을 번호와 함께 출력합니다.
            print(f"  {i}. {q}")

    # 구분선을 출력하여 각 질문-답변 세션을 시각적으로 분리합니다.
    print("\n" + "="*80 + "\n")

    # 결과 딕셔너리를 반환합니다. 이렇게 하면 함수 호출자가 필요한 경우 추가 처리가 가능합니다.
    return result

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: RAG 체인을 사용자 친화적인 인터페이스로 래핑하여,
#    질문을 쉽게 하고 결과를 깔끔하게 표시할 수 있게 합니다.
#
# 2. 함수 정의의 구조:
#    def ask_faq(question): → question 매개변수를 받는 함수 정의
#    """FAQ 챗봇에게 질문하기""" → docstring (함수 설명, help(ask_faq)로 확인 가능)
#
# 3. 함수 내부 실행 흐름:
#    a) qa_chain.invoke(question): RAG 체인 실행
#    b) result에서 answer와 sources 추출
#    c) 포맷팅된 출력 생성
#    d) 결과 반환
#
# 4. 출력 포맷팅의 장점:
#    - 이모지 사용: 시각적 구분과 사용자 경험 향상
#    - 구조화된 정보: 질문, 답변, 참고 문서 수, 관련 FAQ 목록
#    - 가독성: 구분선으로 각 세션 분리
#
# 5. 메타데이터 활용:
#    - doc.metadata.get("question"): Document 객체 생성 시 저장했던 원본 FAQ 질문
#    - 이 정보를 출력함으로써 답변의 출처를 투명하게 보여줌
#    - 사용자는 답변의 근거가 된 FAQ 질문을 확인할 수 있음
#
# 6. 반환값의 활용:
#    - 함수는 result 딕셔너리를 반환하므로, 호출자가 필요한 경우 추가 처리 가능
#    - 예: 답변을 파일에 저장, 추가 분석, 다른 시스템에 전달 등
#
# 7. 함수의 재사용성:
#    - 이 함수를 반복 호출하여 여러 질문을 쉽게 테스트 가능
#    - 다른 스크립트나 애플리케이션에서 임포트하여 사용 가능
#
# 8. 사용 예시:
#    ask_faq("연차 신청 방법 알려줘")
#    ask_faq("월급날이 언제인가요?")
#    ask_faq("재택근무 가능한가요?")
#
# 9. 출력 예시:
#    💬 질문: 연차 신청 방법 알려줘
#
#    🤖 답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다...
#
#    📚 참고한 문서 수: 3
#
#    관련 FAQ:
#      1. 연차는 어떻게 신청하나요?
#      2. 재택근무 신청 방법을 알려주세요
#      3. 경조사 휴가 기준은?
#
#    ================================================================================
#
# 10. 디버깅과 모니터링 지원:
#     - 참고한 문서 수 출력: retriever가 몇 개의 문서를 검색했는지 확인
#     - 관련 FAQ 목록: 어떤 문서가 검색되었는지 확인 (유사도 검색 품질 평가)
#     - LangSmith에서 더 상세한 실행 정보 확인 가능
#
# 11. 확장 가능성:
#     - 이 함수를 수정하여 파일 출력, 로깅, 이메일 전송 등의 기능 추가 가능
#     - GUI 애플리케이션 또는 웹 서비스의 백엔드 함수로 활용 가능
#
# 12. 오류 처리 (현재 버전에는 없지만 추가 가능):
#     - qa_chain.invoke()에서 예외 발생 시 try-except 블록으로 처리
#     - sources가 None일 경우의 처리
#     - answer가 빈 문자열일 경우의 처리
#
# 중요 개념:
# - 함수 캡슐화: 복잡한 RAG 체인 실행을 단순한 함수 호출로 추상화
# - 사용자 인터페이스: 기술적 세부사항을 숨기고 사용자 친화적인 출력 제공
# - 결과 시각화: 텍스트 정보를 구조화하고 이모지로 시각적 강조

In [ ]:
# RAG 챗봇의 성능을 종합적으로 테스트하기 위한 다양한 질문들을 정의합니다.
test_questions = [
    "연차는 어떻게 신청하나요?",  # FAQ에 정확히 존재하는 질문 (직접 매칭)
    "급여일이 언제에요?",         # FAQ에 존재하지만 표현이 약간 다른 질문 (의미적 유사성)
    "재택근무 가능한가요?",       # FAQ에 존재하지만 질문 형식이 다른 경우
    "회사에서 주식 투자 교육을 받을 수 있나요?"  # FAQ에 없는 질문 (RAG 시스템의 한계 테스트)
]
# 각 테스트 질문의 목적:
# 1. 정확한 매칭: FAQ에 존재하는 질문이 정확히 검색되는지 테스트
# 2. 의미적 검색: 다른 표현으로 같은 의미의 질문을 검색하는지 테스트
# 3. 자연어 변형: 더 자연스러운 대화 형식의 질문 처리 테스트
# 4. 한계 테스트: FAQ에 없는 주제에 대한 시스템의 대응 방식 테스트

# test_questions 리스트의 각 질문에 대해 순차적으로 ask_faq 함수를 호출합니다.
for q in test_questions:
    # for 루프: test_questions 리스트의 각 요소(질문 문자열)를 q 변수에 할당하면서 반복
    ask_faq(q)  # 각 질문 q에 대해 ask_faq 함수 호출

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: 구축한 RAG 챗봇 시스템을 다양한 유형의 질문으로 테스트하여
#    시스템의 강점과 약점을 파악하고, 필요시 개선점을 찾습니다.
#
# 2. 테스트 질문들의 전략적 선택:
#    a) "연차는 어떻게 신청하나요?"
#       - 목적: 정확한 키워드 매칭 테스트
#       - FAQ 데이터에 존재하는 동일한 질문
#       - retriever가 정확히 동일한 문서를 검색해야 함
#       - LLM은 FAQ의 답변을 그대로 제공해야 함
#
#    b) "급여일이 언제에요?"
#       - 목적: 유사 표현 검색 테스트
#       - FAQ에는 "월급날은 언제인가요?"로 존재
#       - retriever가 "월급날"과 "급여일"의 의미적 유사성을 이해하는지 테스트
#       - 자연어 처리의 동의어, 유의어 인식 능력 평가
#
#    c) "재택근무 가능한가요?"
#       - 목적: 자연스러운 대화식 질문 처리 테스트
#       - FAQ에는 "재택근무 신청 방법을 알려주세요"로 존재
#       - retriever가 질문의 의도(가능 여부)를 이해하고 관련 문서를 검색하는지 테스트
#       - 간결한 질문 형식에 대한 시스템 반응 평가
#
#    d) "회사에서 주식 투자 교육을 받을 수 있나요?"
#       - 목적: 시스템의 한계와 안전 장치 테스트
#       - FAQ에는 없는 주제 (교육은 있지만 주식 투자 교육은 없음)
#       - retriever가 얼마나 관련 없는 문서를 검색하는지 관찰
#       - LLM이 프롬프트의 지시("FAQ에 없는 내용...")를 따르는지 확인
#
# 3. for 루프의 실행 과정:
#    a) 첫 번째 반복: q = "연차는 어떻게 신청하나요?"
#       - ask_faq("연차는 어떻게 신청하나요?") 호출
#       - 결과 출력
#
#    b) 두 번째 반복: q = "급여일이 언제에요?"
#       - ask_faq("급여일이 언제에요?") 호출
#       - 결과 출력
#
#    c) 세 번째 반복: q = "재택근무 가능한가요?"
#       - ask_faq("재택근무 가능한가요?") 호출
#       - 결과 출력
#
#    d) 네 번째 반복: q = "회사에서 주식 투자 교육을 받을 수 있나요?"
#       - ask_faq("회사에서 주식 투자 교육을 받을 수 있나요?") 호출
#       - 결과 출력
#
# 4. 각 테스트에서 예상되는 결과:
#    a) 첫 번째 질문:
#       - 검색된 문서: "연차는 어떻게 신청하나요?" FAQ (높은 유사도 점수)
#       - 답변: FAQ의 답변을 기반으로 한 정확한 정보
#       - 참고 문서 수: 3 (k=3 설정)
#       - 관련 FAQ 목록에 "연차는 어떻게 신청하나요?" 포함
#
#    b) 두 번째 질문:
#       - 검색된 문서: "월급날은 언제인가요?" FAQ (높은 유사도 점수)
#       - 답변: FAQ의 답변을 기반으로 한 정확한 정보
#       - 참고 문서 수: 3
#       - 관련 FAQ 목록에 "월급날은 언제인가요?" 포함
#
#    c) 세 번째 질문:
#       - 검색된 문서: "재택근무 신청 방법을 알려주세요" FAQ (높은 유사도 점수)
#       - 답변: FAQ의 답변을 기반으로 한 정확한 정보
#       - 참고 문서 수: 3
#       - 관련 FAQ 목록에 "재택근무 신청 방법을 알려주세요" 포함
#
#    d) 네 번째 질문:
#       - 검색된 문서: "사내 교육 프로그램이 있나요?" 등 관련성 낮은 문서
#       - 답변: "정확한 정보는 인사팀(내선 1234)으로 문의해주세요" 또는 유사한 안내
#       - 참고 문서 수: 3 (관련성은 낮을 수 있음)
#       - 관련 FAQ 목록: "사내 교육 프로그램이 있나요?" 등 관련성 낮은 FAQ
#
# 5. 테스트의 중요성:
#    - 회귀 테스트: 시스템 변경 후 기존 기능이 정상 작동하는지 확인
#    - 성능 평가: retriever의 의미론적 검색 능력 평가
#    - 사용자 경험: 실제 사용자가 할 법한 다양한 질문 형식에 대한 대응 평가
#    - 한계 인식: 시스템이 무엇을 할 수 있고 없는지 명확히 이해
#
# 6. 출력 결과 분석 포인트:
#    - 답변의 정확성: FAQ 정보를 정확히 반영하는가?
#    - 답변의 자연스러움: 기계적인 답변인가, 자연스러운가?
#    - 검색의 정확성: 관련성 높은 문서를 검색하는가?
#    - 한계 처리: FAQ에 없는 질문에 적절히 대응하는가?
#
# 7. 확장 가능한 테스트 시나리오:
#    - 동의어 테스트: "휴가 신청", "연차 휴가" 등 다양한 표현
#    - 오타 테스트: "엔차", "월급날" 등 오타 포함 질문
#    - 복합 질문 테스트: "연차 신청과 재택근무 모두 알려주세요"
#    - 맥락적 질문 테스트: "그건 어떻게 하나요?" (이전 대화 맥락 필요)
#
# 8. 실제 실행 시 주의사항:
#    - API 호출 비용: 각 질문당 OpenAI API 호출 발생 (임베딩 + LLM)
#    - 실행 시간: 질문당 수초 소요 (네트워크 지연, API 응답 시간)
#    - LangSmith 로깅: 모든 실행이 LangSmith에 기록됨
#
# 9. 이 테스트를 통해 얻을 수 있는 통찰:
#    - RAG 시스템의 강점: 정확한 정보 제공, 환각 현상 감소
#    - 개선 필요점: retriever의 k 값 조정, 임베딩 모델 변경, 프롬프트 튜닝
#    - 사용자 피드백: 어떤 질문에 대해 사용자가 만족하는지/불만족하는지 파악
#
# 중요 개념:
# - 테스트 주도 개발: 다양한 시나리오로 시스템을 테스트하여 품질 보장
# - 의미론적 검색 평가: 단순 키워드 매칭이 아닌 의미 이해도 평가
# - 시스템 한계 테스트: 시스템의 경계를 명확히 정의하고 관리

💬 질문: 연차는 어떻게 신청하나요?

🤖 답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우에는 전화로 먼저 알리고 사후 처리도 가능합니다. 추가로 궁금한 점이 있으시면 언제든지 문의해 주세요!

📚 참고한 문서 수: 3

관련 FAQ:
  1. 연차는 어떻게 신청하나요?
  2. 주차는 어떻게 하나요?
  3. 재택근무 신청 방법을 알려주세요


💬 질문: 급여일이 언제에요?

🤖 답변: 급여는 매월 25일에 지급됩니다. 만약 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는 급여일 오전에 이메일로 발송됩니다. 추가로 궁금한 사항이 있으시면 언제든지 문의해 주세요!

📚 참고한 문서 수: 3

관련 FAQ:
  1. 월급날은 언제인가요?
  2. 야근 수당은 어떻게 되나요?
  3. 경조사 휴가 기준은?


💬 질문: 재택근무 가능한가요?

🤖 답변: 네, 재택근무는 주 2회까지 가능합니다. 전날 오후 6시까지 팀장에게 슬랙으로 신청하시면 됩니다. 재택근무 중에도 오전 10시부터 오후 5시까지는 업무 가능 상태여야 하며, 화상회의 참석이 가능해야 합니다. 추가로 궁금한 점이 있으시면 언제든지 문의해 주세요!

📚 참고한 문서 수: 3

관련 FAQ:
  1. 재택근무 신청 방법을 알려주세요
  2. 연차는 어떻게 신청하나요?
  3. 주차는 어떻게 하나요?


💬 질문: 회사에서 주식 투자 교육을 받을 수 있나요?

🤖 답변: 현재 제공된 FAQ 정보에는 주식 투자 교육에 대한 내용이 포함되어 있지 않습니다. 정확한 정보는 인사팀(내선 1234)으로 문의해 주시기 바랍니다. 도움이 필요하시면 언제든지 말씀해 주세요!

📚 참고한 문서 수: 3

관련 FAQ:
  1. 사내 교육 프로그램이 있나요?
  2. 건강검진은 언제 받나요?
  3. 복지 포인트는 어떻게 사용하나요?




## Step 5: LangSmith 모니터링

LangSmith에서 다음을 확인할 수 있습니다:
- 전체 체인 실행 과정
- 각 단계별 소요 시간
- 사용한 토큰 수와 비용
- 검색된 문서들
- LLM의 입출력

👉 https://smith.langchain.com 에서 'korean-faq-chatbot' 프로젝트를 확인하세요!

## Step 6: 대화 기록 기능 추가 (Memory)

In [ ]:
# 메모리(대화 기록) 기능을 추가하여 대화형 RAG 챗봇을 구축합니다.
# 이전 대화 맥락을 기억하고 참조할 수 있게 하는 기능입니다.

# operator 모듈에서 itemgetter를 임포트합니다. 딕셔너리에서 특정 키의 값을 추출하는 데 사용됩니다.
from operator import itemgetter

# 대화 기록을 포함한 프롬프트 작성을 위한 클래스들을 임포트합니다.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 대화 기록을 관리하는 Runnable 클래스를 임포트합니다.
from langchain_core.runnables import RunnableWithMessageHistory
# 출력 파서를 임포트합니다.
from langchain_core.output_parsers import StrOutputParser
# 메모리에 대화 기록을 저장하는 클래스를 임포트합니다.
from langchain_core.chat_history import InMemoryChatMessageHistory
# OpenAI 채팅 모델을 임포트합니다.
from langchain_openai import ChatOpenAI

# 1) LLM & retriever 준비
# 새로운 LLM 인스턴스를 생성합니다. (이전에 이미 생성되었더라도 명시적으로 재생성)
llm = ChatOpenAI(model="gpt-4o-mini")  # 주석: 이전에 이미 생성되었다면 생략 가능하지만, 명확성을 위해 재정의
# 벡터 저장소를 검색기로 변환합니다. 이전과 동일한 설정(k=3)을 사용합니다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2) HR FAQ RAG 프롬프트 (대화 기록 + 컨텍스트 + 질문)
# 대화 기록을 포함한 RAG 프롬프트 템플릿을 정의합니다.
rag_template = """당신은 회사의 HR 챗봇입니다.
아래 제공된 FAQ 정보를 바탕으로 직원의 질문에 정확하고 친절하게 답변해주세요.

FAQ 정보:
{context}

질문: {question}

답변 가이드:
1. FAQ에 정확한 답변이 있으면 그대로 전달하세요
2. FAQ에 없는 내용이면 "정확한 정보는 인사팀(내선 1234)으로 문의해주세요"라고 안내하세요
3. 친절하고 전문적인 톤을 유지하세요

답변:"""  # 이 템플릿은 대화 기록 아래에 위치하게 됩니다.

# ChatPromptTemplate.from_messages를 사용하여 계층적 프롬프트를 생성합니다.
# 이 방식은 시스템 메시지, 대화 기록, 사용자 입력을 구조화된 방식으로 결합합니다.
rag_prompt = ChatPromptTemplate.from_messages([
    # 시스템 메시지: 챗봇의 역할과 행동 지침을 정의합니다.
    (
        "system",  # 메시지 유형: system (챗봇의 역할 정의)
        "당신은 회사의 HR 챗봇입니다. "
        "FAQ와 이전 대화를 바탕으로 직원의 질문에 정확하고 친절하게 답변해주세요."
    ),
    # MessagesPlaceholder: 대화 기록이 삽입될 자리표시자입니다.
    # "chat_history"라는 이름으로 참조됩니다.
    MessagesPlaceholder("chat_history"),
    # 사용자 메시지: 실제 질문과 RAG 템플릿이 결합됩니다.
    ("human", rag_template),  # 메시지 유형: human (사용자 입력)
    # 이 구조는 다음과 같은 프롬프트를 생성합니다:
    # 1. 시스템 메시지
    # 2. 이전 대화 기록 (여러 개의 교차 메시지)
    # 3. 현재 사용자 질문과 FAQ 컨텍스트
])

# 3) base_rag_chain: question + chat_history → context 검색 → LLM 호출
# 대화 기록을 고려한 RAG 체인을 구성합니다.
base_rag_chain = (
    # 입력 딕셔너리에서 필요한 값들을 추출하고 처리합니다.
    {
        # "context": 질문만 추출하여 retriever에 전달하고 FAQ 문서 검색
        # itemgetter("question")은 입력 딕셔너리에서 "question" 키의 값을 추출합니다.
        # 파이프(|) 연산자로 retriever와 연결하여 질문으로 문서 검색을 수행합니다.
        "context": itemgetter("question") | retriever,
        # "question": 입력 딕셔너리에서 "question" 키의 값을 그대로 추출합니다.
        "question": itemgetter("question"),
        # "chat_history": 입력 딕셔너리에서 "chat_history" 키의 값을 추출합니다.
        "chat_history": itemgetter("chat_history"),
    }
    # 위에서 처리된 결과를 rag_prompt에 전달하여 프롬프트를 완성합니다.
    | rag_prompt
    # 완성된 프롬프트를 LLM에 전달합니다.
    | llm
    # LLM의 응답을 문자열로 파싱합니다.
    | StrOutputParser()
)
# base_rag_chain은 {"question": 질문, "chat_history": 대화기록} 형식의 입력을 기대합니다.
# 출력은 답변 문자열입니다.

# 4) 세션별 대화 히스토리 저장소
# 세션별로 대화 기록을 저장할 딕셔너리를 생성합니다.
store = {}  # 키: session_id, 값: InMemoryChatMessageHistory 객체

def get_session_history(session_id: str):
    """세션 ID에 해당하는 대화 기록을 반환합니다. 없으면 생성합니다."""
    if session_id not in store:
        # 새로운 세션일 경우, InMemoryChatMessageHistory 객체를 생성하여 저장합니다.
        store[session_id] = InMemoryChatMessageHistory()
    # 세션에 해당하는 대화 기록 객체를 반환합니다.
    return store[session_id]
# 이 함수는 RunnableWithMessageHistory에 의해 호출되어 세션별 기록을 관리합니다.

# 5) 대화형 체인으로 래핑
# RunnableWithMessageHistory를 사용하여 대화 기록 기능을 추가합니다.
conversational_chain = RunnableWithMessageHistory(
    # 기본 RAG 체인: 대화 기록을 고려하여 답변을 생성하는 체인
    base_rag_chain,
    # 대화 기록을 가져오는 함수: 세션 ID를 입력으로 받아 해당 세션의 기록을 반환
    get_session_history,
    # 입력에서 사용자 질문이 들어있는 필드 이름
    input_messages_key="question",     # 입력 딕셔너리에서 질문을 찾을 키
    # 프롬프트에서 대화 기록 자리표시자의 이름
    history_messages_key="chat_history",  # 프롬프트의 MessagesPlaceholder 이름과 일치
    # RunnableWithMessageHistory의 작동 방식:
    # 1. 사용자 입력(question)과 세션 ID를 받음
    # 2. get_session_history(session_id)로 해당 세션의 기록 가져옴
    # 3. {"question": 질문, "chat_history": 대화기록} 형태로 base_rag_chain에 전달
    # 4. base_rag_chain이 답변 생성
    # 5. 새 질문과 답변을 대화 기록에 추가하여 다음 대화에서 사용
)

print("대화 기록 기능이 추가된 챗봇이 준비되었습니다!")

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: 대화 기록(메모리) 기능을 추가하여 챗봇이 이전 대화 맥락을 기억하고
#    이를 바탕으로 더 자연스러운 대화를 할 수 있게 합니다.
#
# 2. 대화 기록의 중요성:
#    - 참조적 표현 이해: "그것", "저것", "위에서 말한" 등의 대명사 참조 이해
#    - 맥락 유지: 이전에 논의한 내용을 기억하고 계속된 대화 가능
#    - 개인화: 사용자별 대화 기록을 별도로 관리
#
# 3. MessagesPlaceholder의 역할:
#    - 프롬프트 템플릿 내에 대화 기록이 삽입될 위치를 표시
#    - 실제 실행 시 chat_history에 저장된 메시지들로 대체됨
#    - 메시지는 (role, content) 형식의 튜플 리스트 (예: [("human", "첫 질문"), ("ai", "첫 답변")])
#
# 4. InMemoryChatMessageHistory:
#    - 메모리에 대화 기록을 저장하는 간단한 구현체
#    - 실제 애플리케이션에서는 데이터베이스나 파일 시스템에 저장하는 구현체 사용 가능
#    - 메서드: add_user_message(), add_ai_message(), messages 속성
#
# 5. 세션 관리:
#    - store 딕셔너리: 세션 ID를 키로 하여 각 세션의 대화 기록 저장
#    - 세션 ID: 사용자 식별자, 대화 방 식별자 등으로 구분
#    - 예: "user-123", "chat-room-456", "anonymous-001"
#
# 6. RunnableWithMessageHistory의 내부 작동:
#    a) conversational_chain.invoke(
#           {"question": "새 질문"},
#           config={"configurable": {"session_id": "user-123"}}
#       )
#    b) get_session_history("user-123") 호출 → 해당 세션의 대화 기록 가져옴
#    c) base_rag_chain에 {"question": "새 질문", "chat_history": 기록} 전달
#    d) base_rag_chain 실행 및 답변 생성
#    e) 새 질문과 답변이 해당 세션의 기록에 자동으로 추가됨
#
# 7. 대화 기록이 검색에 미치는 영향:
#    - 현재 구현: retriever에는 오직 현재 질문만 전달됨 (대화 기록은 컨텍스트 검색에 사용되지 않음)
#    - 개선 가능성: 대화 기록을 고려한 검색 질문 재구성 가능
#    - 예: "이전에 연차 신청 방법을 물었는데, 최소 몇 일 전에 신청해야 하나요?" → "연차 신청 최소 기간"
#
# 8. 프롬프트 구조 분석:
#    - 시스템 메시지: 챗봇의 정체성과 기본 지침
#    - 대화 기록: 이전 대화의 전체 흐름
#    - 현재 질문 + FAQ 컨텍스트: 현재 질문에 대한 구체적 정보
#    - LLM은 이 모든 정보를 종합하여 맥락을 이해하고 답변 생성
#
# 9. 대화형 챗봇의 장점:
#    - 자연스러운 대화: "위에서 말한 건 어떻게 하나요?" 같은 후속 질문 가능
#    - 사용자 경험 향상: 매번 맥락을 반복 설명할 필요 없음
#    - 효율성: 짧고 간결한 질문으로도 정확한 답변 가능
#
# 10. 주의사항:
#     - 토큰 제한: 대화 기록이 길어지면 프롬프트 토큰 수 초과 가능
#     - 해결책: 최근 N개 메시지만 유지, 요약, 토큰 수 제한 등
#     - 개인정보: 대화 기록에 민감한 정보가 포함될 수 있음
#
# 11. 확장 가능성:
#     - 영구 저장: 데이터베이스에 대화 기록 저장
#     - 기록 요약: 긴 대화 기록을 요약하여 토큰 절약
#     - 다중 세션: 한 사용자의 여러 대화 세션 관리
#
# 중요 개념:
# - 대화 기록(Chat History): 사용자와 AI의 메시지 교환 기록
# - 세션 관리(Session Management): 사용자별 또는 대화별 기록 분리 관리
# - 맥락 보존(Context Preservation): 대화의 흐름과 참조 정보 유지
# - 메모리(Memory): LangChain에서 대화 기록 관리 기능을 지칭하는 용어

✅ 대화 기록 기능이 추가된 챗봇이 준비되었습니다!


In [ ]:
# 6) chat() 함수 정의
# 대화형 HR FAQ 챗봇을 사용하기 위한 편의 함수를 정의합니다.
def chat(question, session_id="demo-user"):
    """대화형 HR FAQ 챗봇 질의 함수"""
    # conversational_chain을 호출하여 질문에 대한 답변을 생성합니다.
    # 이 체인은 대화 기록을 자동으로 관리합니다.
    result = conversational_chain.invoke(
        # 입력 데이터: 질문과 빈 대화 기록을 제공합니다.
        # 실제 대화 기록은 RunnableWithMessageHistory가 내부적으로 관리하므로,
        # 여기서는 빈 리스트를 전달하더라도 체인이 세션 저장소에서 실제 기록을 불러옵니다.
        {"question": question, "chat_history": []},  # chat_history는 RunnableWithMessageHistory가 내부 처리
        # 설정(config): 세션 관리를 위한 설정을 제공합니다.
        config={"configurable": {"session_id": session_id}},
        # configurable 설정 설명:
        # - session_id: 대화 기록을 식별하기 위한 고유 식별자
        # - 기본값은 "demo-user"로, 지정하지 않으면 모든 테스트가 동일한 세션으로 처리됨
        # - 다른 session_id를 지정하면 별도의 대화 기록이 유지됨
    )
    # 결과를 사용자 친화적인 형식으로 출력합니다.
    print(f"💬 질문: {question}")
    print(f"🤖 답변: {result}\n")
    # 생성된 답변을 반환합니다. 필요시 추가 처리를 할 수 있도록 합니다.
    return result
    # 함수 실행 과정 상세:
    # 1. conversational_chain.invoke() 호출
    # 2. RunnableWithMessageHistory가 get_session_history(session_id)를 호출하여 대화 기록 가져옴
    # 3. base_rag_chain에 {"question": 질문, "chat_history": 실제기록} 전달
    # 4. base_rag_chain 실행:
    #    a) retriever가 현재 질문으로 FAQ 검색
    #    b) rag_prompt가 시스템 메시지 + 대화 기록 + 현재 질문+FAQ로 프롬프트 완성
    #    c) LLM이 답변 생성
    #    d) StrOutputParser가 문자열로 변환
    # 5. 새 질문과 답변이 대화 기록에 추가됨
    # 6. 결과 출력 및 반환


# 7) 연속 대화 테스트
# chat() 함수를 사용하여 대화 기록 기능이 제대로 작동하는지 테스트합니다.
# 첫 번째 질문: 연차 신청 방법에 대해 묻습니다.
chat("연차는 어떻게 신청하나요?")
# 이 호출에서:
# - session_id를 지정하지 않았으므로 기본값 "demo-user" 사용
# - "demo-user" 세션의 대화 기록은 처음이므로 빈 상태
# - 질문에 대한 답변 생성 후, 질문과 답변이 세션 기록에 저장됨

# 두 번째 질문: 이전 대화 맥락을 참조하는 후속 질문을 합니다.
chat("최소 몇 일 전에 신청해야 하나요?")  # 이전 맥락 따라가야 함
# 이 호출에서:
# - 동일한 session_id("demo-user")를 사용하므로 이전 대화 기록이 존재함
# - 대화 기록에는 첫 번째 질문과 답변이 포함되어 있음
# - LLM은 "최소 몇 일 전에 신청해야 하나요?"라는 질문을 볼 때,
#   대화 기록을 통해 이전에 연차 신청에 대해 이야기했음을 인지함
# - 따라서 "최소 몇 일 전"이 연차 신청의 최소 기간을 묻는 것임을 이해함
# - FAQ 검색은 여전히 현재 질문만으로 수행되지만, 프롬프트에 대화 기록이 포함되어 맥락 이해에 도움

# 세 번째 질문: 완전히 새로운 주제로 전환합니다.
chat("급여는 언제 받나요?")            # 새로운 주제
# 이 호출에서:
# - 세션 기록에는 이전 두 번의 대화(연차 신청 관련)가 모두 포함됨
# - LLM은 대화 기록을 보지만, 현재 질문이 새로운 주제임을 인지함
# - FAQ 검색은 "급여" 관련 문서를 찾음
# - 대화 기록이 있지만 새로운 주제이므로 이전 맥락과 혼동하지 않음

# --------------------------
# 테스트 시나리오의 목적과 예상 결과:
# --------------------------
# 1. 첫 번째 질문 테스트 목적:
#    - 기본적인 RAG 기능이 작동하는지 확인
#    - 세션 기록이 처음 시작될 때 빈 상태에서도 정상 작동하는지 확인
#    - 예상: 연차 신청 방법에 대한 정확한 답변 출력
#
# 2. 두 번째 질문 테스트 목적:
#    - 대화 기록 기능이 작동하는지 확인
#    - 이전 맥락을 이해하고 참조적 질문에 올바르게 답변하는지 확인
#    - 예상: "연차 신청은 최소 3일 전에 신청해야 합니다"와 같은 답변
#    - 중요: 두 번째 질문만으로는 불완전하지만, 대화 기록을 통해 맥락 파악 가능
#
# 3. 세 번째 질문 테스트 목적:
#    - 대화 기록이 새로운 주제로 전환될 때 혼란을 주지 않는지 확인
#    - 이전 주제와 무관한 새로운 질문에 대해 독립적으로 답변하는지 확인
#    - 예상: 급여일에 대한 정확한 답변 (이전 연차 대화와 혼동 없음)
#
# 4. 대화 기록 관리의 실제 동작:
#    - 각 chat() 호출 후, 해당 세션의 기록은 자동으로 업데이트됨
#    - "demo-user" 세션의 기록은 다음과 같이 성장함:
#        호출 1 후: [(human, "연차는 어떻게..."), (ai, "연차 신청은...")]
#        호출 2 후: 위 기록 + [(human, "최소 몇 일..."), (ai, "최소 3일...")]
#        호출 3 후: 위 기록 + [(human, "급여는..."), (ai, "급여는...")]
#
# 5. 세션 ID의 중요성:
#    - 동일한 session_id를 사용하면 대화 기록이 공유됨
#    - 다른 session_id를 사용하면 완전히 별도의 대화 기록이 유지됨
#    - 예: chat("질문", session_id="user1")과 chat("질문", session_id="user2")는 독립적
#
# 6. 대화 기록의 프롬프트 내 역할:
#    - rag_prompt 구조에서 MessagesPlaceholder("chat_history") 위치에 삽입됨
#    - LLM은 시스템 메시지 → 대화 기록 → 현재 질문+FAQ 순으로 정보를 받음
#    - 이를 통해 LLM은 전체 대화 흐름을 이해하고 맥락에 맞는 답변 생성
#
# 7. 테스트에서 확인할 수 있는 대화형 챗봇의 장점:
#    - 자연스러운 대화 흐름: 참조적 표현 사용 가능
#    - 사용자 편의성: 매번 전체 맥락을 반복할 필요 없음
#    - 일관성: 동일한 세션에서 일관된 정보 제공
#
# 8. 잠재적 문제점 및 고려사항:
#    - 토큰 제한: 대화 기록이 길어지면 프롬프트가 너무 커질 수 있음
#    - 정보 과부하: 오래된 대화 기록이 현재 질문과 무관할 수 있음
#    - 세션 관리: 적절한 세션 만료 정책 필요
#
# 9. 출력 예시:
#    💬 질문: 연차는 어떻게 신청하나요?
#    🤖 답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다...
#
#    💬 질문: 최소 몇 일 전에 신청해야 하나요?
#    🤖 답변: 연차는 최소 3일 전에 신청해야 합니다. (이전 답변의 정보를 참조하여)
#
#    💬 질문: 급여는 언제 받나요?
#    🤖 답변: 급여는 매월 25일에 지급됩니다...
#
# 10. 실제 애플리케이션에서의 활용:
#     - 웹 챗봇: 사용자별 세션 ID를 부여하여 개인별 대화 기록 관리
#     - 음성 비서: 대화 기반 상호작용에서 맥락 유지
#     - 고객 지원: 이전 상담 내역을 참조하여 일관된 지원 제공
#
# 중요 개념:
# - 세션 기반 대화: 사용자 또는 대화 단위로 기록을 분리 관리
# - 참조 해결(Reference Resolution): 대명사나 불완전한 표현의 의미를 맥락에서 파악
# - 대화 상태 관리(Dialogue State Tracking): 대화의 현재 상태와 흐름을 추적

💬 질문: 연차는 어떻게 신청하나요?
🤖 답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우에는 전화로 먼저 알리고 사후 처리도 가능합니다. 추가로 궁금한 사항이 있다면 언제든지 물어보세요!

💬 질문: 최소 몇 일 전에 신청해야 하나요?
🤖 답변: 연차는 최소 3일 전에 신청해야 합니다. 추가로 궁금한 사항이 있으시면 언제든지 문의해 주세요!

💬 질문: 급여는 언제 받나요?
🤖 답변: 급여는 매월 25일에 지급됩니다. 만약 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는 급여일 오전에 이메일로 발송됩니다. 추가로 궁금한 사항이 있으시면 언제든지 문의해 주세요!



'급여는 매월 25일에 지급됩니다. 만약 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는 급여일 오전에 이메일로 발송됩니다. 추가로 궁금한 사항이 있으시면 언제든지 문의해 주세요!'

## Step 7: 성능 평가 및 개선

### 7.1 평가 데이터셋 만들기

In [ ]:
# RAG 시스템의 성능을 평가하기 위한 평가 데이터셋과 평가 함수를 정의합니다.
# 평가 데이터셋: 질문과 기대되는 답변의 핵심 키워드들로 구성됩니다.

# 평가용 질문-답변 쌍을 정의합니다.
# 각 항목은 딕셔너리로 구성되며, 질문과 기대 키워드 목록을 포함합니다.
eval_data = [
    {
        "question": "연차 신청은 어떻게 하나요?",  # 평가할 질문 (FAQ에 존재)
        "expected_keywords": ["그룹웨어", "전자결재", "3일 전", "팀장 승인"]  # 답변에 포함되어야 할 핵심 키워드들
        # 이 키워드들은 FAQ 답변의 핵심 정보를 대표합니다.
        # "그룹웨어": 신청 플랫폼
        # "전자결재": 구체적인 메뉴
        # "3일 전": 시간 제약
        # "팀장 승인": 필요한 승인 절차
    },
    {
        "question": "월급은 언제 들어오나요?",  # 평가할 질문 (FAQ에 존재하지만 표현이 다름)
        "expected_keywords": ["25일", "영업일", "급여명세서"]  # 기대 키워드들
        # "25일": 급여 지급일
        # "영업일": 주말/공휴일 대체 기준
        # "급여명세서": 관련 문서
    },
    {
        "question": "재택 근무가 가능한가요?",  # 평가할 질문 (자연스러운 표현)
        "expected_keywords": ["주 2회", "슬랙", "전날 오후 6시"]  # 기대 키워드들
        # "주 2회": 빈도 제한
        # "슬랙": 신청 방법
        # "전날 오후 6시": 신청 마감 시간
    }
]
# 평가 데이터 선정 기준:
# 1. FAQ에 존재하는 질문
# 2. 다양한 표현 방식 (직접적, 간접적, 자연스러운)
# 3. 핵심 정보가 명확한 답변
# 4. 키워드 기반 평가가 가능한 정도의 구체성

# 간단한 평가 함수를 정의합니다.
# 이 함수는 생성된 답변에 기대 키워드들이 얼마나 포함되어 있는지 계산합니다.
def evaluate_answer(question, answer, expected_keywords):
    """
    생성된 답변을 평가하여 기대 키워드 포함 여부를 점수화합니다.

    매개변수:
        question (str): 평가 질문 (함수 내부에서 사용되지 않지만 맥락 유지용)
        answer (str): 평가할 답변 문자열
        expected_keywords (list): 답변에 포함되길 기대하는 키워드 문자열들의 리스트

    반환값:
        tuple: (포함된 키워드 수, 전체 키워드 수)
    """
    # 리스트 컴프리헨션과 sum() 함수를 사용하여 포함된 키워드 수를 계산합니다.
    # 1. for keyword in expected_keywords: 각 기대 키워드에 대해
    # 2. if keyword in answer: 키워드가 답변 문자열에 포함되어 있는지 확인
    # 3. 1 for ...: 포함되어 있으면 1, 아니면 0
    # 4. sum(): 모든 1들의 합계 계산 = 포함된 키워드 수
    score = sum(1 for keyword in expected_keywords if keyword in answer)

    # 전체 키워드 수를 계산합니다.
    total = len(expected_keywords)

    # 점수(포함된 키워드 수)와 전체 키워드 수를 튜플로 반환합니다.
    return score, total
    # 예시: answer에 4개 중 3개의 키워드가 포함되었다면 (3, 4) 반환

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: RAG 시스템의 성능을 정량적으로 평가하기 위한 도구를 제공합니다.
#    키워드 기반 평가는 간단하지만 효과적인 평가 방법입니다.
#
# 2. 평가 데이터(eval_data) 설계의 의도:
#    a) 다양한 질문 유형 포함:
#       - 정확한 매칭: "연차 신청은 어떻게 하나요?" (FAQ와 거의 동일)
#       - 자연스러운 변형: "월급은 언제 들어오나요?" (FAQ: "월급날은 언제인가요?")
#       - 간결한 질문: "재택 근무가 가능한가요?" (FAQ: "재택근무 신청 방법을 알려주세요")
#
#    b) 키워드 선정 기준:
#       - 핵심 정보: 답변의 가장 중요한 정보를 대표
#       - 구체성: 모호하지 않고 명확한 단어
#       - 독립성: 다른 맥락에서 자주 등장하지 않는 단어
#       - 필수성: 이 정보가 없으면 답변이 불완전한 단어
#
# 3. evaluate_answer 함수의 작동 원리:
#    a) 입력: 질문(참고용), 답변(평가 대상), 기대 키워드 리스트
#    b) 처리: 각 키워드가 답변 문자열에 포함되어 있는지 확인
#    c) 출력: (포함된 키워드 수, 전체 키워드 수)
#
#    d) 키워드 매칭 방식:
#       - 대소문자 구분: 한국어는 대소문자 구분이 없으므로 문제 없음
#       - 부분 일치: "그룹웨어"가 "사내 그룹웨어 시스템"에 포함되면 매칭됨
#       - 정확도: 단순 포함 여부만 확인하므로 정확한 의미 파악은 못함
#
# 4. 이 평가 방법의 장점:
#    - 구현이 간단하고 실행 속도가 빠름
#    - 명확한 기준으로 일관된 평가 가능
#    - 특정 정보의 포함 여부를 정확히 확인 가능
#    - 자동화된 테스트에 적합
#
# 5. 이 평가 방법의 한계:
#    - 의미론적 이해 부족: 동의어나 유의어를 인식하지 못함
#       예: "그룹웨어"와 "인사시스템"이 같은 의미라도 매칭 안 됨
#    - 맥락 고려 부족: 키워드는 있지만 잘못된 맥락에서 사용될 수 있음
#       예: "3일 전에 취소해야 합니다" (잘못된 정보)
#    - 문장 구조 무시: 키워드가 있지만 부정문이나 조건문일 수 있음
#       예: "3일 전에 신청하지 않아도 됩니다" (반대 의미)
#
# 6. 평가 함수의 실제 실행 예시:
#    answer = "연차 신청은 사내 그룹웨어의 전자결재 메뉴에서 합니다. 최소 3일 전에 신청하세요."
#    expected_keywords = ["그룹웨어", "전자결재", "3일 전", "팀장 승인"]
#
#    평가 과정:
#    - "그룹웨어" in answer? → True (1점)
#    - "전자결재" in answer? → True (1점)
#    - "3일 전" in answer? → True (1점)
#    - "팀장 승인" in answer? → False (0점)
#    - 총점: 3, 전체: 4 → (3, 4) 반환
#    - 정확도: 75%
#
# 7. 더 정교한 평가 방법들 (향후 고려 가능):
#    - BLEU 점수: 기계 번역 평가에서 사용되는 n-gram 기반 평가
#    - ROUGE 점수: 요약 평가에서 사용되는 재현율 기반 평가
#    - 의미적 유사도: 임베딩 벡터의 코사인 유사도 계산
#    - LLM 기반 평가: GPT-4 등으로 답변의 품질을 평가
#
# 8. 평가 데이터 확장 방안:
#    - 더 많은 질문 추가 (다양성 확보)
#    - 부정적 테스트 케이스 (잘못된 정보가 생성되지 않는지)
#    - 맥락적 질문 (대화 기록이 필요한 질문)
#    - FAQ에 없는 질문 (적절한 대응을 하는지)
#
# 9. 이 평가 시스템의 실제 활용:
#    - 개발 중 품질 검증
#    - 모델 변경 시 성능 비교
#    - 하이퍼파라미터 튜닝 시 지표
#    - 지속적인 통합/배포(CI/CD) 파이프라인에 통합
#
# 10. 다음 단계에서 실제 평가를 실행하기 위해 필요한 것:
#     - 실제로 qa_chain을 사용하여 각 평가 질문에 대한 답변 생성
#     - 생성된 답변을 evaluate_answer 함수로 평가
#     - 전체 정확도 계산 및 결과 출력
#
# 중요 개념:
# - 평가 메트릭(Evaluation Metric): 시스템 성능을 측정하기 위한 지표
# - 키워드 매칭(Keyword Matching): 특정 단어나 구의 포함 여부로 평가
# - 정량적 평가(Quantitative Evaluation): 수치화된 점수로 성능 측정
# - 벤치마크(Benchmark): 표준화된 테스트를 통한 성능 비교

In [ ]:
# RAG 챗봇의 성능 평가를 실행하고 결과를 출력합니다.

# 평가 시작을 알리는 헤더를 출력합니다.
print(" 챗봇 성능 평가\n" + "="*50)
# "="*50은 문자열 곱셈으로 50개의 "=" 문자를 생성하여 구분선을 만듭니다.

# 전체 평가 결과를 누적하기 위한 변수들을 초기화합니다.
total_score = 0        # 모든 질문에서 포함된 키워드의 총 개수
total_keywords = 0     # 모든 질문의 전체 키워드 수의 합

# 평가 데이터(eval_data)의 각 항목에 대해 반복 평가를 수행합니다.
for eval_item in eval_data:
    # 각 평가 항목은 딕셔너리 형태로 {"question": 질문, "expected_keywords": [키워드들]}입니다.

    # qa_chain을 호출하여 현재 평가 질문에 대한 답변을 생성합니다.
    # 주석: 이전 LangChain 버전에서는 {"query": 질문} 형식이었지만,
    # 현재 버전에서는 질문 문자열을 직접 전달할 수 있습니다.
    result = qa_chain.invoke(eval_item["question"])
    # qa_chain.invoke()의 반환값은 {"answer": 답변문자열, "source_documents": 문서리스트} 형식입니다.

    # 결과에서 답변 문자열을 추출합니다.
    # 주석: 이전 버전에서는 result["result"]였지만, 현재는 result["answer"]입니다.
    answer = result["answer"]

    # evaluate_answer 함수를 호출하여 생성된 답변을 평가합니다.
    # 함수는 (포함된 키워드 수, 전체 키워드 수) 튜플을 반환합니다.
    score, total = evaluate_answer(
        eval_item["question"],        # 평가 질문
        answer,                       # 생성된 답변
        eval_item["expected_keywords"] # 기대 키워드 목록
    )

    # 전체 누적 점수에 현재 질문의 점수를 더합니다.
    total_score += score
    # 전체 누적 키워드 수에 현재 질문의 전체 키워드 수를 더합니다.
    total_keywords += total

    # 현재 질문에 대한 평가 결과를 출력합니다.
    print(f"\n질문: {eval_item['question']}")  # 평가 질문 출력
    print(f"점수: {score}/{total} (포함된 키워드 비율)")  # 점수 출력 (예: 3/4)
    # 답변의 처음 100자만 출력합니다 (너무 길면 가독성이 떨어지므로).
    print(f"답변: {answer[:100]}...")  # 답변의 처음 100자와 "..." 출력

# 모든 평가가 끝난 후 전체 정확도를 계산합니다.
# 정확도 = (포함된 키워드 총 수 / 전체 키워드 총 수) * 100
accuracy = (total_score / total_keywords) * 100

# 최종 결과를 출력합니다.
print("\n" + "="*50)  # 구분선 출력
print(f"전체 정확도: {accuracy:.1f}%")  # 전체 정확도 출력 (소수점 1자리까지)

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: 구축한 RAG 챗봇의 성능을 정량적으로 평가하고,
#    개별 질문별 성능과 전체 평균 성능을 확인합니다.
#
# 2. 평가 실행의 상세 흐름:
#    a) 첫 번째 평가 항목 (연차 신청):
#       1) qa_chain.invoke("연차 신청은 어떻게 하나요?") 실행
#       2) retriever가 관련 FAQ 검색, LLM이 답변 생성
#       3) evaluate_answer 호출: 답변에 키워드 포함 여부 확인
#       4) 점수 누적 및 결과 출력
#
#    b) 두 번째 평가 항목 (월급):
#       1) qa_chain.invoke("월급은 언제 들어오나요?") 실행
#       2) retriever가 "월급날은 언제인가요?" FAQ 검색 (의미적 유사성 활용)
#       3) evaluate_answer 호출: 키워드 매칭 확인
#       4) 점수 누적 및 결과 출력
#
#    c) 세 번째 평가 항목 (재택 근무):
#       1) qa_chain.invoke("재택 근무가 가능한가요?") 실행
#       2) retriever가 "재택근무 신청 방법을 알려주세요" FAQ 검색
#       3) evaluate_answer 호출: 키워드 매칭 확인
#       4) 점수 누적 및 결과 출력
#
# 3. 출력 결과 예시:
#     챗봇 성능 평가
#    ==================================================
#
#    질문: 연차 신청은 어떻게 하나요?
#    점수: 4/4 (포함된 키워드 비율)
#    답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다. 최소 3일 전...
#
#    질문: 월급은 언제 들어오나요?
#    점수: 3/3 (포함된 키워드 비율)
#    답변: 급여는 매월 25일에 지급됩니다. 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는...
#
#    질문: 재택 근무가 가능한가요?
#    점수: 3/3 (포함된 키워드 비율)
#    답변: 재택근무는 주 2회까지 가능합니다. 전날 오후 6시까지 팀장에게 슬랙으로 신청하면 됩니다...
#
#    ==================================================
#    전체 정확도: 100.0%
#
# 4. 평가 결과 해석:
#    - 개별 질문 점수: 각 질문에 대해 얼마나 정확한 정보를 제공하는지 확인
#    - 전체 정확도: 시스템의 전반적인 성능을 한눈에 파악
#    - 답변 미리보기: 답변의 품질과 자연스러움을 대략적으로 확인
#
# 5. 정확도 계산의 수학적 이해:
#    - 예시: 3개 질문, 각각 4, 3, 3개의 키워드 (총 10개 키워드)
#    - 포함된 키워드: 4, 3, 3개 → 총 10개
#    - 정확도: (10 / 10) * 100 = 100%
#
# 6. 이 평가의 의미와 한계:
#    - 의미: RAG 시스템이 FAQ 정보를 정확히 전달하는지 확인
#    - 한계: 키워드 존재 여부만 확인하므로 의미 왜곡, 문장 구조 등은 평가하지 못함
#    - 보완: 실제 사용자 테스트, LLM 기반 평가, A/B 테스트 등과 병행 필요
#
# 7. 실제 평가 시 고려사항:
#    - API 비용: 각 평가 질문당 OpenAI API 호출 발생
#    - 실행 시간: 평가에 소요되는 총 시간 (질문 수 * 질문당 처리 시간)
#    - 일관성: 동일한 질문에 대해 여러 번 평가하여 일관성 확인 가능
#
# 8. 평가 결과를 통한 개선점 발견:
#    - 낮은 점수 질문: retriever가 관련 문서를 찾지 못하거나 LLM이 정보를 잘 추출하지 못함
#    - 개선 방안: retriever의 k 값 조정, 임베딩 모델 변경, 프롬프트 수정
#    - 예시: "팀장 승인" 키워드가 빠졌다면, 해당 정보가 FAQ에 있지만 LLM이 누락했을 수 있음
#
# 9. 확장 가능한 평가 시나리오:
#    - 대화 기록 포함 평가: 연속적인 질문에 대한 평가
#    - 다양한 사용자 시나리오: 실제 사용자의 질문 패턴을 반영한 평가
#    - 부정 테스트: 잘못된 정보를 생성하지 않는지 평가
#    - 성능 벤치마크: 다른 모델이나 설정과의 비교 평가
#
# 10. 이 평가 코드의 재사용성:
#     - 다른 FAQ 데이터셋에 적용 가능
#     - 다른 RAG 시스템과 비교 평가에 사용 가능
#     - CI/CD 파이프라인에 통합하여 자동화된 품질 검증 가능
#
# 중요 개념:
# - 성능 평가(Performance Evaluation): 시스템의 품질을 객관적으로 측정하는 과정
# - 정량적 지표(Quantitative Metrics): 수치화된 성능 측정값
# - 자동화된 테스트(Automated Testing): 코드를 통해 반복적이고 일관된 테스트 수행
# - 품질 보증(Quality Assurance): 시스템이 요구사항을 만족하는지 확인하는 활동

 챗봇 성능 평가

질문: 연차 신청은 어떻게 하나요?
점수: 3/4 (포함된 키워드 비율)
답변: 연차 신청은 사내 그룹웨어의 '전자결재' 메뉴에서 '휴가신청서'를 작성하시면 됩니다. 최소 3일 전에 신청해야 하며, 팀장의 승인이 필요합니다. 긴급한 경우에는 전화로 먼저 알리고...

질문: 월급은 언제 들어오나요?
점수: 3/3 (포함된 키워드 비율)
답변: 급여는 매월 25일에 지급됩니다. 만약 25일이 주말이나 공휴일인 경우, 그 전 영업일에 지급됩니다. 급여명세서는 급여일 오전에 이메일로 발송됩니다. 추가적인 질문이 있으시면 언제...

질문: 재택 근무가 가능한가요?
점수: 3/3 (포함된 키워드 비율)
답변: 네, 재택근무는 주 2회까지 가능합니다. 전날 오후 6시까지 팀장에게 슬랙으로 신청하시면 됩니다. 재택근무 중에도 오전 10시부터 오후 5시까지는 업무 가능 상태여야 하며, 화상회...

전체 정확도: 90.0%


## Step 8: 실전 배포를 위한 추가 기능

In [ ]:
# Gradio 라이브러리를 설치합니다. Gradio는 머신러닝 모델이나 파이썬 함수를 위한 웹 기반 인터페이스를 빠르게 생성할 수 있는 라이브러리입니다.
# 공식 웹사이트: https://www.gradio.app/

# 명령어 앞의 느낌표(!)는 Colab이나 Jupyter 노트북에서 셀을 시스템 명령어로 실행하도록 합니다.
!pip install gradio

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: RAG 챗봇을 웹 인터페이스를 통해 접근 가능하게 만들기 위해 Gradio를 설치합니다.
#
# 2. pip install의 작동 원리:
#    a) !pip install gradio 명령어가 실행되면 Python 패키지 관리자(pip)가 활성화됩니다.
#    b) pip는 Python Package Index(PyPI)에서 gradio 패키지를 찾습니다.
#    c) 패키지와 그 의존성(dependencies)들을 다운로드하여 현재 파이썬 환경에 설치합니다.
#    d) 설치 과정에는 다음이 포함될 수 있습니다:
#        - gradio 라이브러리 파일 다운로드
#        - 필요한 추가 패키지 설치 (예: markdown, requests, numpy 등)
#        - 설치 진행 상황을 터미널에 표시
#
# 3. Gradio 라이브러리의 주요 특징:
#    - 사용하기 쉬운 웹 인터페이스 생성
#    - 실시간 상호작용 지원
#    - 자동으로 생성되는 공유 가능한 링크 (Colab이나 로컬 서버에서)
#    - 다양한 컴포넌트 지원 (텍스트 박스, 버튼, 이미지, 오디오 등)
#
# 4. 설치 과정에서 발생할 수 있는 일:
#    - 이미 설치된 경우: "Requirement already satisfied" 메시지 출력
#    - 새로 설치하는 경우: 다운로드 및 설치 진행 상황 표시
#    - 의존성 충돌: 다른 패키지와의 버전 충돌이 발생할 수 있음
#    - 네트워크 문제: 인터넷 연결 문제로 설치 실패 가능
#
# 5. 설치 후 사용 가능한 기능:
#    - import gradio: 파이썬 코드에서 Gradio 임포트
#    - gr.Interface(): 간단한 함수를 웹 인터페이스로 변환
#    - gr.ChatInterface(): 챗봇 인터페이스 생성
#    - 다양한 레이아웃과 스타일링 옵션
#
# 6. 이 설치 명령어가 필요한 이유:
#    - Colab 환경에는 기본적으로 Gradio가 설치되어 있지 않을 수 있습니다.
#    - 로컬 환경에서도 명시적으로 설치해야 사용 가능합니다.
#    - 패키지 버전을 최신으로 유지하여 새로운 기능 사용 가능.
#
# 7. 대체 설치 방법들:
#    - 특정 버전 설치: !pip install gradio==3.x.x
#    - 개발 버전 설치: !pip install gradio@git+https://github.com/gradio-app/gradio
#    - conda를 사용한 설치: !conda install -c conda-forge gradio
#
# 8. 설치 확인 방법:
#    - 다음 셀에서 import gradio가 오류 없이 실행되면 성공적으로 설치된 것입니다.
#    - 또는 !pip show gradio 명령어로 설치 정보 확인 가능.
#
# 9. 설치 시 주의사항:
#    - 가상환경 사용 시 해당 가상환경에 설치되어야 함
#    - 패키지 충돌을 피하기 위해 requirements.txt 파일 관리 권장
#    - 프로덕션 환경에서는 버전을 명시적으로 고정하는 것이 좋음
#
# 10. 다음 단계에서 Gradio를 사용하여:
#     - 챗봇을 웹 인터페이스로 감싸기
#     - 사용자가 브라우저에서 직접 질문할 수 있게 하기
#     - 실시간으로 답변을 받을 수 있는 UI 제공
#     - 예시 질문 버튼 등 사용자 편의 기능 추가
#
# 중요 개념:
# - 패키지 관리자(Package Manager): 소프트웨어 패키지의 설치, 업그레이드, 제거를 관리하는 도구
# - 의존성(Dependencies): 한 패키지가 작동하기 위해 필요한 다른 패키지들
# - 가상환경(Virtual Environment): 프로젝트별로 독립적인 Python 환경을 생성하는 도구
# - 웹 인터페이스(Web Interface): 웹 브라우저를 통해 접근할 수 있는 사용자 인터페이스

In [ ]:
# Gradio 라이브러리를 임포트합니다. 일반적으로 gr로 별칭을 지정하여 사용합니다.
import gradio as gr

# Gradio 챗봇 인터페이스용 함수를 정의합니다.
# 이 함수는 Gradio의 ChatInterface에서 사용되며, (message, history) 두 개의 매개변수를 받습니다.
def chatbot_interface(message, history):
    """Gradio 챗봇 인터페이스"""
    # Gradio는 대화 기록(history)을 자체적으로 관리하므로,
    # 우리는 qa_chain을 사용하여 현재 메시지에 대한 답변만 생성합니다.

    # qa_chain을 호출하여 메시지에 대한 답변을 생성합니다.
    # message는 사용자가 입력한 현재 질문입니다.
    result = qa_chain.invoke(message)
    # qa_chain의 반환값: {"answer": 답변문자열, "source_documents": 문서리스트}

    # 답변 문자열을 추출합니다.
    answer = result["answer"]

    # 참고 문서(검색된 문서)에서 FAQ 질문 제목을 추출합니다.
    # 리스트 컴프리헨션을 사용하여 각 문서의 메타데이터에서 "question" 값을 가져옵니다.
    sources = [
        doc.metadata.get("question", "N/A")  # "question" 키가 없으면 "N/A" 반환
        for doc in result.get("source_documents", [])  # source_documents가 없으면 빈 리스트 사용
    ]

    # 검색된 문서가 있으면, 답변에 참고한 FAQ 목록을 추가합니다.
    if sources:
        # 답변에 구분선과 "참고한 FAQ:" 헤더를 추가합니다.
        answer += "\n\n📚 참고한 FAQ:\n" + "\n".join(
            [f"- {s}" for s in sources[:2]]  # 최대 2개의 FAQ만 표시 (가독성을 위해)
        )

    # 답변 문자열을 반환합니다. Gradio는 이를 챗봇 응답으로 표시합니다.
    return answer

# Gradio ChatInterface를 생성합니다. 이는 챗봇 인터페이스를 쉽게 만들 수 있는 고수준 컴포넌트입니다.
demo = gr.ChatInterface(
    # fn: 챗봇 로직을 구현한 함수
    fn=chatbot_interface,
    # title: 웹 인터페이스의 제목
    title="🏢 HR FAQ 챗봇",
    # description: 웹 인터페이스의 설명
    description="회사 HR 관련 질문을 해보세요!",
    # examples: 사용자가 클릭할 수 있는 예시 질문들
    examples=[
        "연차는 어떻게 신청하나요?",
        "급여일이 언제인가요?",
        "재택근무가 가능한가요?",
        "건강검진은 언제 받나요?"
    ]
    # ChatInterface의 추가 옵션들 (기본값 사용):
    # - theme: 기본 테마
    # - textbox: 입력창 디자인
    # - submit_btn: 전송 버튼
    # - retry_btn, undo_btn, clear_btn: 대화 관리 버튼들
)

# Gradio 인터페이스를 실행합니다.
demo.launch(share=True)
# share=True: 공개적으로 접근 가능한 URL을 생성합니다 (Colab에서 유용).
# Colab 환경에서 이 코드를 실행하면 Gradio는 로컬 서버를 시작하고 공유 링크를 제공합니다.

# --------------------------
# 이 코드 블록의 목적과 작동 원리:
# --------------------------
# 1. 목적: RAG 챗봇을 웹 기반 인터페이스로 제공하여,
#    기술적 지식이 없는 사용자도 쉽게 접근하고 사용할 수 있게 합니다.
#
# 2. Gradio의 작동 원리:
#    - 백그라운드에서 로컬 웹 서버를 시작합니다 (보통 localhost:7860).
#    - 웹 페이지를 생성하여 사용자가 브라우저에서 접근할 수 있게 합니다.
#    - 사용자 입력을 받아 Python 함수로 전달하고, 결과를 웹 페이지에 표시합니다.
#
# 3. chatbot_interface 함수의 역할:
#    - Gradio와 RAG 시스템 간의 브리지 역할
#    - 입력: 사용자 메시지와 Gradio가 관리하는 대화 기록
#    - 출력: 생성된 답변 문자열
#    - Gradio는 이 함수를 호출하여 각 사용자 메시지에 대한 응답을 생성
#
# 4. Gradio ChatInterface의 주요 기능:
#    - 자동 대화 기록 관리: 이전 메시지들을 화면에 표시
#    - 예시 질문: 클릭 한 번으로 질문 입력 가능
#    - 응답 스트리밍: 실시간으로 답변이 생성되는 것처럼 보이게 할 수 있음
#    - 다국어 지원: 다양한 언어 인터페이스
#    - 반응형 디자인: 모바일과 데스크톱에서 모두 작동
#
# 5. launch(share=True)의 의미:
#    - Colab에서 실행 시: 공개 URL 생성 (예: https://xxxx.gradio.app)
#    - 로컬에서 실행 시: localhost:7860에서 접근 가능
#    - share=True 없이 실행: 로컬에서만 접근 가능
#    - 공유 링크는 72시간 동안 유효 (Gradio의 무료 호스팅)
#
# 6. 웹 인터페이스의 구성 요소:
#    a) 헤더: 제목과 설명
#    b) 채팅 영역: 사용자와 AI의 메시지가 표시되는 곳
#    c) 입력창: 사용자가 질문을 입력하는 곳
#    d) 전송 버튼: 질문을 제출하는 버튼
#    e) 예시 질문: 클릭하여 입력창에 자동으로 입력되는 버튼들
#    f) 대화 관리 버튼: 다시하기, 되돌리기, 지우기
#
# 7. 실제 실행 시 발생하는 일:
#    a) demo.launch()가 호출되면 Gradio 서버가 시작됨
#    b) Colab에서는 공유 링크가 생성되어 출력됨
#    c) 링크를 클릭하면 웹 브라우저에서 챗봇 인터페이스 열림
#    d) 사용자가 질문을 입력하면:
#        1) 브라우저 → Gradio 서버 → chatbot_interface 함수
#        2) chatbot_interface → qa_chain → RAG 시스템
#        3) RAG 시스템 → chatbot_interface → Gradio 서버 → 브라우저
#        4) 사용자가 답변을 화면에서 확인
#
# 8. 소스 문서 표시의 중요성:
#    - 투명성: 답변의 근거를 보여줌으로써 신뢰성 향상
#    - 디버깅: 어떤 FAQ가 참조되었는지 확인 가능
#    - 학습: 사용자가 관련 FAQ를 추가로 확인할 수 있음
#    - 제한: sources[:2]로 최대 2개만 표시하여 가독성 유지
#
# 9. 이 웹 인터페이스의 장점:
#    - 접근성: 웹 브라우저만 있으면 누구나 사용 가능
#    - 사용 편의성: 복잡한 설정이나 명령어 없이 직관적 사용
#    - 실시간 상호작용: 질문과 답변이 실시간으로 표시
#    - 공유 가능: 링크를 통해 다른 사람과 쉽게 공유
#    - 무료 호스팅: Gradio의 공유 기능으로 별도 서버 없이 배포 가능
#
# 10. 확장 가능한 기능들:
#     - 파일 업로드: 사용자가 문서를 업로드하여 질문 가능
#     - 음성 입력: 음성으로 질문하고 텍스트로 답변 받기
#     - 다중 사용자: 동시에 여러 사용자가 접속 가능
#     - 테마 변경: 어두운 모드, 색상 변경 등
#     - 로깅: 사용자 질문과 답변을 로그로 저장
#
# 11. 주의사항:
#     - API 비용: 많은 사용자가 접속하면 OpenAI API 비용 증가
#     - 보안: 공개 URL은 누구나 접근 가능하므로 민감한 정보 처리 주의
#     - 성능: 동시 접속자가 많으면 응답 시간 지연 가능
#     - 세션 관리: 웹 인터페이스는 별도의 세션 관리가 필요 없음 (Gradio가 처리)
#
# 12. 실제 배포를 위한 고려사항:
#     - share=True는 개발용이며, 프로덕션에서는 별도 서버에 배포 권장
#     - 도메인 연결, HTTPS 설정, 사용자 인증 등 추가 구현 필요
#     - 모니터링: LangSmith와 연동하여 사용 패턴 모니터링
#     - 스케일링: 사용자 증가에 따른 서버 자원 확장 계획
#
# 중요 개념:
# - 웹 애플리케이션 인터페이스: 사용자가 웹 브라우저를 통해 상호작용할 수 있는 인터페이스
# - 프론트엔드/백엔드 분리: Gradio가 프론트엔드, RAG 시스템이 백엔드 역할
# - 실시간 통신: 사용자 입력과 시스템 응답이 실시간으로 주고받아짐
# - 공유 링크: 인터넷을 통해 애플리케이션을 임시로 공유하는 방법

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e98e5bb708d65b364.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 추가 학습 자료

1. **LangChain 공식 문서**: https://python.langchain.com/
2. **LangSmith 가이드**: https://docs.smith.langchain.com/
3. **RAG 심화**: https://python.langchain.com/docs/use_cases/question_answering/
4. **프롬프트 엔지니어링**: https://www.promptingguide.ai/
